In [42]:
import warnings
warnings.filterwarnings('ignore')

import os
import pickle

import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

from pandas.tseries.offsets import MonthEnd, MonthBegin
from maricovault.MaricoDB import MaricoSnowflake

from joblib import Parallel, delayed

In [43]:
base_dir = '/data/aman_singh/acuuracy_check'
os.chdir(base_dir)

In [44]:
# Define variables

run_months_list = ['2026-07-31']

In [45]:
def get_dbconnection(db_name): 

    KEY_VAULT_NAME = "prod-pwd"
    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection

In [46]:
dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


### Helper functions

In [47]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

realignment_df = realignment_df[
    realignment_df['channel'].isin(['QCOM', 'QCOM B2C', 'ALL'])]


def realign_pskus(data, column):
    realignment_data = realignment_df.copy()
    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']

    
    data[column] = data[column].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data[column] == old_psku, column
        ] = new_psku

    return data

In [48]:
from tqdm import tqdm

def impute_missing_dates(
        df,
        freq='M',
        key=['CHAIN', 'FACILITY_NAME', 'PARENT_MATERIAL_CODE'], 
        date_col='MONTH_DATE',
        max_date='2026-12-31'
    ):

    impute_df = df.copy()
    impute_df[date_col] = pd.to_datetime(impute_df[date_col])
    impute_df['key'] = impute_df[key].astype(str).agg('_'.join, axis=1)
    min_dates_df = impute_df.groupby(
        'key', as_index=False
    )[date_col].min()

    def impute_missing_dates_key(key, min_date, max_date):
        df_imputed = pd.DataFrame(
            pd.date_range(min_date, max_date, freq=freq),
            columns=[date_col]
        )
        df_imputed['key'] = key
        return df_imputed
    
    outputs = Parallel(n_jobs=-1)(
        delayed(impute_missing_dates_key)(row['key'], row[date_col], max_date)
        for idx, row in tqdm(min_dates_df.iterrows())
    )
    df_full = pd.concat(outputs)
    df_full.reset_index(drop=True, inplace=True)

    df_full = df_full.merge(
        impute_df, on=['key', date_col], how='left',
    )

    return df_full

In [49]:
non_rc_to_rc_mapping = {
    718948: 722208,
    718356: 721837,
    718455: 721963,
    718341: 721898,
    718342: 721836,
    719008: 721842,
    729186: 729165,
    718288: 721843,
    722301: 722302
}

rc_to_non_rc_mapping = {value: key for key, value in non_rc_to_rc_mapping.items()}

rc_skus = list(rc_to_non_rc_mapping.keys()) + [729280]
non_rc_skus = list(rc_to_non_rc_mapping.values())

In [50]:
def get_final_psku(x):
    if x['zone'] in ['North', 'East']:
        if (x['rc_psku'] == 1):
            return x['parent_material_code']
        else:
            return non_rc_to_rc_mapping[x['parent_material_code']]
    else:
        if (x['rc_psku'] == 0):
            return x['parent_material_code']
        else:
            return rc_to_non_rc_mapping[x['parent_material_code']]   
        
            
def zepto_saff_golf_fix(df):

    df = df.copy()

    zepto_gold_df = df[
        (df['chain'] == 'Zepto') &
        (df['material_group_code'] == 'SAFF GOLD')
    ]

    rest = df[
        ~((df['chain'] == 'Zepto') &
        (df['material_group_code'] == 'SAFF GOLD'))
    ]

    ########### Offtake City to New City ###########
    city_mappings_df = pd.read_excel(
        r'/data/aman_singh/mt_forecast/City Mappings QCOM.xlsb', 
        'Final'
    )

    city_mappings_df.rename(columns={'City': 'warehouse_city', 'Final City': 'final'}, inplace=True)

    city_mappings_df.columns = city_mappings_df.columns.str.lower()

    assert city_mappings_df.duplicated(subset=['platform_name', 'city']).sum() == 0

    city_mappings_df.rename(columns={'platform_name': 'chain'}, inplace=True)
    #####################################################################


    #### New City to Zone Mappings #######
    # region_mappings = pd.read_excel(
    #     r"C:\Users\aniket.patel\OneDrive - Marico Ltd\Zepto - SAFF GOLD - Region Mappings.xlsx"    
    # )
    # region_mappings.columns = region_mappings.columns.str.lower()
    data_region = {
        "city": [
            "Faridabad", "Ahmedabad", "Jaipur", "Patiala", "Pune",
            "Bangalore", "GURGAON", "Medak", "Chennai", "Lucknow",
            "Hyderabad", "Jhajjar", "Howrah", "Mumbai", "Bhiwandi"
        ],
        "State/Region": [
            "Haryana", "Gujarat", "Rajasthan", "Punjab", "Maharashtra",
            "Karnataka", "Haryana", "Telangana", "Tamil Nadu", "Uttar Pradesh",
            "Telangana", "Haryana", "West Bengal", "Maharashtra", "Maharashtra"
        ],
        "zone": [
            "North", "West", "North", "North", "West",
            "South", "North", "South", "South", "North",
            "South", "North", "East", "West", "West"
        ]
    }

    region_mappings = pd.DataFrame(data_region)
    #######################################


    ### FIX ####
    len_before_merge = len(zepto_gold_df)
    zepto_gold_df = zepto_gold_df.merge(
        city_mappings_df[['chain', 'city', 'final']],
        on=['chain', 'city'],
        how='left'
    )
    assert len_before_merge == len(zepto_gold_df)
    del len_before_merge

    zepto_gold_df = zepto_gold_df[zepto_gold_df['final'].notna()]

    len_before_merge = len(zepto_gold_df)
    zepto_gold_df = zepto_gold_df.merge(
        region_mappings[['city', 'zone']].rename(
            columns={'city': 'final'}
        ),
        how='left', 
        on=['final']
    )
    assert len_before_merge == len(zepto_gold_df)
    del len_before_merge


    zepto_gold_df['rc_psku'] = zepto_gold_df['parent_material_code'].map(
        lambda x: 1 if x in rc_skus else 0
    )

    zepto_gold_df['final_psku'] = zepto_gold_df.apply(lambda x: get_final_psku(x), axis=1)

    zepto_gold_df = zepto_gold_df.groupby(
        ['chain', 'city', 'final_psku', 'material_group_code', 
        'month_date'], as_index=False
    )['vol_in_roum'].sum()
    zepto_gold_df.rename(columns={'final_psku': 'parent_material_code'}, inplace=True)

    df = pd.concat([rest, zepto_gold_df], ignore_index=True)

    return df

### Primary Actuals + Sec Plan Data

In [51]:
plan_actuals_query = """
-- Primary base table
SELECT 
    CASE
        WHEN MCM.chain = 'Kiranakart Technologies' THEN 'Zepto'
        WHEN MCM.chain = 'Grofers' THEN 'Blinkit'
        ELSE MCM.chain
    END AS chain,
    DCM.marico_depot,
    MM.parent_material_code,
    MM.material_group_code,
    MESR.month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain IN ('Grofers', 'Swiggy', 'Zepto', 'Kiranakart Technologies')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
LEFT JOIN 
(
    SELECT DISTINCT
        customer,
        marico_depot
    FROM
        dev_db.data_science.trn_soh_dc_master
) DCM on MESR.distributor_code = DCM.customer
WHERE month_date > '2023-12-31'
GROUP BY 1, 2, 3, 4, 5
ORDER BY 1, 2, 4, 3, 5
"""

plan_actuals_df = pd.read_sql(
    plan_actuals_query,
    prod_conn
)

In [52]:
plan_actuals_df.columns = plan_actuals_df.columns.str.lower()
plan_actuals_df['month_date'] = pd.to_datetime(plan_actuals_df['month_date'])

In [53]:
plan_actuals_df.duplicated(subset=['chain', 'marico_depot', 'parent_material_code', 
     'material_group_code', 'month_date']).sum()

0

In [54]:
plan_actuals_df[plan_actuals_df['marico_depot'].isna()]#['sec_actuals_vol_rum'].sum()

,chain,marico_depot,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
356090,Blinkit,None,718472,ADV-AHO-R,2024-01-01,0.0,0.108491,0.000,0.0
356091,Blinkit,None,718472,ADV-AHO-R,2024-02-01,0.0,0.163414,0.000,0.0
356092,Blinkit,None,718472,ADV-AHO-R,2024-12-31,0.0,0.000000,2.281,0.0
356093,Blinkit,None,718473,ADV-AHO-R,2024-01-01,0.0,0.187036,0.000,0.0
356094,Blinkit,None,718473,ADV-AHO-R,2024-02-01,0.0,0.281743,0.000,0.0
...,...,...,...,...,...,...,...,...,...
742391,Zepto,None,810179,SW_SGPRF,2024-10-01,0.0,11.425066,0.000,0.0
742392,Zepto,None,810179,SW_SGPRF,2024-11-01,0.0,0.466896,0.000,0.0
742393,Zepto,None,810179,SW_SGPRF,2024-12-01,0.0,0.746376,0.000,0.0
742394,Zepto,None,810179,SW_SGPRF,2025-01-01,0.0,0.319030,0.000,0.0


In [55]:
plan_actuals_df.loc[plan_actuals_df['marico_depot'].isna(), 'marico_depot'] = np.nan

In [56]:
plan_actuals_df['marico_depot'] = plan_actuals_df['marico_depot'].str.lower()

In [57]:
plan_actuals_df.shape

(742396, 9)

In [58]:
plan_actuals_df = realign_pskus(plan_actuals_df.copy(), column='parent_material_code')

In [59]:
plan_actuals_df

,chain,marico_depot,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,Blinkit,d112,718472,ADV-AHO-R,2024-01-01,0.0,0.053040,0.0,0.0
1,Blinkit,d112,718472,ADV-AHO-R,2024-02-01,0.0,0.149204,0.0,0.0
2,Blinkit,d112,718472,ADV-AHO-R,2024-03-01,0.0,0.108107,0.0,0.0
3,Blinkit,d112,718473,ADV-AHO-R,2024-01-01,0.0,0.137200,0.0,0.0
4,Blinkit,d112,718473,ADV-AHO-R,2024-02-01,0.0,0.385826,0.0,0.0
...,...,...,...,...,...,...,...,...,...
742391,Zepto,NaN,810179,SW_SGPRF,2024-10-01,0.0,11.425066,0.0,0.0
742392,Zepto,NaN,810179,SW_SGPRF,2024-11-01,0.0,0.466896,0.0,0.0
742393,Zepto,NaN,810179,SW_SGPRF,2024-12-01,0.0,0.746376,0.0,0.0
742394,Zepto,NaN,810179,SW_SGPRF,2025-01-01,0.0,0.319030,0.0,0.0


In [60]:
plan_actuals_df['month_date'] = plan_actuals_df['month_date'] + MonthEnd(0)

In [61]:
plan_actuals_df.duplicated(subset=['chain', 'marico_depot', 'parent_material_code', 
     'material_group_code', 'month_date']).sum()

498794

In [62]:
plan_actuals_df = plan_actuals_df.groupby(
    ['chain', 'marico_depot', 'parent_material_code', 
     'material_group_code', 'month_date'], as_index=False, dropna=False
).sum()

In [63]:
plan_actuals_df['parent_material_code'] = plan_actuals_df['parent_material_code'].astype(int)

In [64]:
plan_actuals_df = plan_actuals_df[plan_actuals_df['material_group_code']!='PABABY_SP']

In [ ]:
plan_actuals_df = impute_missing_dates(
    plan_actuals_df.copy(),
    key=['chain', 'marico_depot', 'parent_material_code'],
    date_col='month_date'
)

12304it [00:03, 4644.15it/s]

In [ ]:
plan_actuals_df.sort_values(by=['key', 'month_date'], inplace=True)

In [ ]:
cols = ['chain', 'marico_depot', 'parent_material_code', 'material_group_code']

plan_actuals_df[cols] = plan_actuals_df.groupby('key')[cols].transform(lambda x: x.ffill().bfill())

In [ ]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    plan_actuals_df[col] = plan_actuals_df[col].fillna(0)

In [ ]:
plan_actuals_df['parent_material_code'] = plan_actuals_df['parent_material_code'].astype(int)

In [ ]:
plan_actuals_df.duplicated(subset=['key', 'month_date']).sum()

483

In [ ]:
plan_actuals_df

,month_date,key,chain,marico_depot,parent_material_code,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,2024-01-31,Blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.001866,0.000,0.0
1,2024-02-29,Blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.003372,0.001,0.0
2,2024-03-31,Blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.002172,0.000,0.0
3,2024-04-30,Blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.004476,0.000,0.0
4,2024-05-31,Blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.003584,0.000,0.0
...,...,...,...,...,...,...,...,...,...,...
562124,2026-08-31,Zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0
562125,2026-09-30,Zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0
562126,2026-10-31,Zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0
562127,2026-11-30,Zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0


In [ ]:
plan_actuals_df[plan_actuals_df['marico_depot'].isna()]

,month_date,key,chain,marico_depot,parent_material_code,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
230402,2024-01-31,Blinkit_nan_718287,Blinkit,NaN,718287,PCNO(R),0.0,0.000409,0.0,0.0
230403,2024-02-29,Blinkit_nan_718287,Blinkit,NaN,718287,PCNO(R),0.0,0.000410,0.0,0.0
230404,2024-03-31,Blinkit_nan_718287,Blinkit,NaN,718287,PCNO(R),0.0,0.000000,0.0,0.0
230405,2024-04-30,Blinkit_nan_718287,Blinkit,NaN,718287,PCNO(R),0.0,0.000000,0.0,0.0
230406,2024-05-31,Blinkit_nan_718287,Blinkit,NaN,718287,PCNO(R),0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
562124,2026-08-31,Zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.0,0.0
562125,2026-09-30,Zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.0,0.0
562126,2026-10-31,Zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.0,0.0
562127,2026-11-30,Zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.0,0.0


In [ ]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if plan_actuals_df[col].min() < 0:
        print(col)

pri_actuals_vol_rum
sec_actuals_vol_rum


In [ ]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    plan_actuals_df[col] = plan_actuals_df[col].clip(lower=0)

In [ ]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if plan_actuals_df[col].min() < 0:
        print(col)

In [ ]:
plan_actuals_df

,month_date,key,chain,marico_depot,parent_material_code,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,2024-01-31,Blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.001866,0.000,0.0
1,2024-02-29,Blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.003372,0.001,0.0
2,2024-03-31,Blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.002172,0.000,0.0
3,2024-04-30,Blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.004476,0.000,0.0
4,2024-05-31,Blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.003584,0.000,0.0
...,...,...,...,...,...,...,...,...,...,...
562124,2026-08-31,Zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0
562125,2026-09-30,Zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0
562126,2026-10-31,Zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0
562127,2026-11-30,Zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0


In [ ]:
plan_actuals_df.sort_values(by=['key', 'month_date'], inplace=True)

In [ ]:
plan_actuals_df.duplicated(subset=['key', 'month_date']).sum()

483

In [ ]:
plan_actuals_df[plan_actuals_df.duplicated(subset = ['key','month_date'], keep = False)].sort_values(by = 
                                                    ['key','month_date'])['material_group_code'].unique()

array(['LIVON S-R', 'LVNPST_ML', 'PABABY_GM', 'PADV_WIPS'], dtype=object)

In [ ]:
plan_actuals_df[plan_actuals_df['material_group_code'].isin(['LVNPST_ML', 'PABABY_GM', 'PADV_WIPS'])]['pri_actuals_vol_rum'].sum()

27513.111

In [ ]:
plan_actuals_df = plan_actuals_df[~plan_actuals_df['material_group_code'].isin(['LVNPST_ML', 'PABABY_GM', 'PADV_WIPS'])]

In [ ]:
plan_actuals_df['Primary P3M'] = plan_actuals_df.groupby(
    ['key'], 
    as_index = False, group_keys = False
)['pri_actuals_vol_rum'].shift(1).rolling(window=3, min_periods=1).mean()

In [ ]:
plan_actuals_df.sample(1)

,month_date,key,chain,marico_depot,parent_material_code,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,Primary P3M
487222,2026-08-31,Zepto_d231_809046,Zepto,d231,809046,PABABY_GM,0.0,0.0,0.0,0.0,0.0


In [ ]:
plan_actuals_df

,month_date,key,chain,marico_depot,parent_material_code,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,Primary P3M
0,2024-01-31,Blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.001866,0.000,0.0,NaN
1,2024-02-29,Blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.003372,0.001,0.0,0.0
2,2024-03-31,Blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.002172,0.000,0.0,0.0
3,2024-04-30,Blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.004476,0.000,0.0,0.0
4,2024-05-31,Blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.003584,0.000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
581056,2026-08-31,Zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0,0.0
581057,2026-09-30,Zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0,0.0
581058,2026-10-31,Zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0,0.0
581059,2026-11-30,Zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0,0.0


In [ ]:
plan_actuals_df.groupby(
    ['chain', 'marico_depot','parent_material_code', 
     'material_group_code', 'month_date'], as_index=False, dropna=False
).sum().to_csv('plan_actuals_aggregated3.csv', index=False)

In [ ]:
plan_actuals_df[plan_actuals_df['month_date'] == '2026-04-30']['sec_actuals_vol_rum'].sum()

137224.646

### Offtakes

In [ ]:
offtakes_monthly_query = """
SELECT
    OTM.platform_name AS chain,
    OTM.city,
    MM.parent_material_code,
    MM.material_group_code,
    OTM.month_date,
    SUM(CASE
        WHEN MM.uom_reporting IN ('KL', 'TO') THEN ROUND(OTM.vol_in_lit / POW(10, 3), 4)
        ELSE OTM.vol_in_lit
    END) AS vol_in_roum
FROM 
    dwh_ecommplatform_offtake OTM
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON OTM.material_code = MM.material_code
WHERE
    OTM.platform_name IN ('Blinkit', 'Swiggy', 'Zepto') AND
    OTM.month_date > '2022-12-31' 
GROUP BY 1, 2, 3, 4, 5
ORDER BY 1, 2, 3, 5, 4
""" 

offtakes_monthly_df = pd.read_sql(
    offtakes_monthly_query,
    prod_conn
)

In [ ]:
offtakes_monthly_df.columns = offtakes_monthly_df.columns.str.lower()
offtakes_monthly_df['month_date'] = pd.to_datetime(offtakes_monthly_df['month_date'])
offtakes_monthly_df.head()

,chain,city,parent_material_code,material_group_code,month_date,vol_in_roum
0,Blinkit,Abohar,718398,SAFF ACTV,2026-05-31,0.0130
1,Blinkit,Abohar,718434,PADVJAS-R,2026-05-31,0.3000
2,Blinkit,Abohar,718553,PA-BDYLOT,2026-05-31,0.4000
3,Blinkit,Abohar,718559,SFOATS-FL,2026-05-31,0.0001
4,Blinkit,Abohar,718681,PADV-HRCR,2026-05-31,0.1000


In [ ]:
offtakes_monthly_df['month_date'].max()

Timestamp('2026-05-31 00:00:00')

In [ ]:
offtakes_monthly_df['parent_material_code'] = offtakes_monthly_df['parent_material_code'].astype(int)

In [ ]:
offtakes_monthly_df

,chain,city,parent_material_code,material_group_code,month_date,vol_in_roum
0,Blinkit,Abohar,718398,SAFF ACTV,2026-05-31,0.0130
1,Blinkit,Abohar,718434,PADVJAS-R,2026-05-31,0.3000
2,Blinkit,Abohar,718553,PA-BDYLOT,2026-05-31,0.4000
3,Blinkit,Abohar,718559,SFOATS-FL,2026-05-31,0.0001
4,Blinkit,Abohar,718681,PADV-HRCR,2026-05-31,0.1000
...,...,...,...,...,...,...
656086,Zepto,Warangal,810519,SAF_CDPRS,2025-10-31,0.0010
656087,Zepto,Warangal,810519,SAF_CDPRS,2025-11-30,0.0010
656088,Zepto,Warangal,810520,SAF_CDPRS,2025-10-31,0.0020
656089,Zepto,Warangal,810520,SAF_CDPRS,2025-12-31,0.0010


In [ ]:
offtakes_monthly_df = zepto_saff_golf_fix(offtakes_monthly_df.copy())

In [ ]:
offtakes_monthly_df['run_month'] = offtakes_monthly_df['month_date'] + MonthEnd(0)

In [ ]:
offtakes_monthly_df

,chain,city,parent_material_code,material_group_code,month_date,vol_in_roum,run_month
0,Blinkit,Abohar,718398,SAFF ACTV,2026-05-31,0.0130,2026-05-31
1,Blinkit,Abohar,718434,PADVJAS-R,2026-05-31,0.3000,2026-05-31
2,Blinkit,Abohar,718553,PA-BDYLOT,2026-05-31,0.4000,2026-05-31
3,Blinkit,Abohar,718559,SFOATS-FL,2026-05-31,0.0001,2026-05-31
4,Blinkit,Abohar,718681,PADV-HRCR,2026-05-31,0.1000,2026-05-31
...,...,...,...,...,...,...,...
656082,Zepto,Warangal,729186,SAFF GOLD,2025-06-30,0.0030,2025-06-30
656083,Zepto,Warangal,729186,SAFF GOLD,2025-07-31,0.0030,2025-07-31
656084,Zepto,Warangal,729186,SAFF GOLD,2025-08-31,0.0060,2025-08-31
656085,Zepto,Warangal,729186,SAFF GOLD,2025-10-31,0.0030,2025-10-31


In [ ]:
zepto_mapping = pd.read_excel('/data/aman_singh/mt_forecast/Q-com Depot-FC-City Mapping v2.0.xlsx', sheet_name = 'Zepto')
zepto_mapping = zepto_mapping[zepto_mapping['Status'] == 'Active']
zepto_mapping = zepto_mapping[['Channel','City', 'Marico Depot']].drop_duplicates()
zepto_mapping.rename(columns = {'Channel':'platform_name', 'City':'city', 'Marico Depot':'depot'}, inplace = True)
zepto_mapping['platform_name'] = zepto_mapping['platform_name'].str.lower()
zepto_mapping['city'] = zepto_mapping['city'].str.lower()

blinkit_mapping = pd.read_excel('/data/aman_singh/mt_forecast/Q-com Depot-FC-City Mapping v2.0.xlsx', sheet_name = 'BlinkIt')
blinkit_mapping = blinkit_mapping[['Channel','City', 'Marico Depot']].drop_duplicates()
blinkit_mapping.rename(columns = {'Channel':'platform_name', 'City':'city', 'Marico Depot':'depot'}, inplace = True)
blinkit_mapping['platform_name'] = blinkit_mapping['platform_name'].str.lower()
blinkit_mapping['city'] = blinkit_mapping['city'].str.lower()

swiggy_mapping = pd.read_excel('/data/aman_singh/mt_forecast/Q-com Depot-FC-City Mapping v2.0.xlsx', sheet_name = 'Swiggy')
swiggy_mapping = swiggy_mapping[['Channel','City', 'Marico Depot']].drop_duplicates()
swiggy_mapping.rename(columns = {'Channel':'platform_name', 'City':'city', 'Marico Depot':'depot'}, inplace = True)
swiggy_mapping['platform_name'] = swiggy_mapping['platform_name'].str.lower()
swiggy_mapping['city'] = swiggy_mapping['city'].str.lower()

depot_city_mapping = pd.concat([zepto_mapping, blinkit_mapping, swiggy_mapping])
depot_city_mapping[depot_city_mapping.duplicated(subset=['platform_name', 'city'], keep = False)]


,platform_name,city,depot


In [ ]:
y = offtakes_monthly_df.copy()
#offtake_df = x.copy()

In [ ]:
offtakes_monthly_df

,chain,city,parent_material_code,material_group_code,month_date,vol_in_roum,run_month
0,Blinkit,Abohar,718398,SAFF ACTV,2026-05-31,0.0130,2026-05-31
1,Blinkit,Abohar,718434,PADVJAS-R,2026-05-31,0.3000,2026-05-31
2,Blinkit,Abohar,718553,PA-BDYLOT,2026-05-31,0.4000,2026-05-31
3,Blinkit,Abohar,718559,SFOATS-FL,2026-05-31,0.0001,2026-05-31
4,Blinkit,Abohar,718681,PADV-HRCR,2026-05-31,0.1000,2026-05-31
...,...,...,...,...,...,...,...
656082,Zepto,Warangal,729186,SAFF GOLD,2025-06-30,0.0030,2025-06-30
656083,Zepto,Warangal,729186,SAFF GOLD,2025-07-31,0.0030,2025-07-31
656084,Zepto,Warangal,729186,SAFF GOLD,2025-08-31,0.0060,2025-08-31
656085,Zepto,Warangal,729186,SAFF GOLD,2025-10-31,0.0030,2025-10-31


In [ ]:
depot_city_mapping.rename(columns = {'platform_name':'chain'}, inplace = True)

In [ ]:

offtakes_monthly_df['chain'] = offtakes_monthly_df['chain'].str.lower()
offtakes_monthly_df['city'] = offtakes_monthly_df['city'].str.lower()
len_before_merge = len(offtakes_monthly_df)
offtakes_monthly_df = offtakes_monthly_df.merge(depot_city_mapping, on = ['chain', 'city'], how = 'left')
assert len_before_merge == len(offtakes_monthly_df)
offtakes_monthly_df


,chain,city,parent_material_code,material_group_code,month_date,vol_in_roum,run_month,depot
0,blinkit,abohar,718398,SAFF ACTV,2026-05-31,0.0130,2026-05-31,NaN
1,blinkit,abohar,718434,PADVJAS-R,2026-05-31,0.3000,2026-05-31,NaN
2,blinkit,abohar,718553,PA-BDYLOT,2026-05-31,0.4000,2026-05-31,NaN
3,blinkit,abohar,718559,SFOATS-FL,2026-05-31,0.0001,2026-05-31,NaN
4,blinkit,abohar,718681,PADV-HRCR,2026-05-31,0.1000,2026-05-31,NaN
...,...,...,...,...,...,...,...,...
656082,zepto,warangal,729186,SAFF GOLD,2025-06-30,0.0030,2025-06-30,D530
656083,zepto,warangal,729186,SAFF GOLD,2025-07-31,0.0030,2025-07-31,D530
656084,zepto,warangal,729186,SAFF GOLD,2025-08-31,0.0060,2025-08-31,D530
656085,zepto,warangal,729186,SAFF GOLD,2025-10-31,0.0030,2025-10-31,D530


In [ ]:
offtakes_monthly_df.isnull().sum()

chain                       0
city                        0
parent_material_code        0
material_group_code         0
month_date                  0
vol_in_roum                 0
run_month                   0
depot                   10003
dtype: int64

In [ ]:
offtakes_monthly_df

,chain,city,parent_material_code,material_group_code,month_date,vol_in_roum,run_month,depot
0,blinkit,abohar,718398,SAFF ACTV,2026-05-31,0.0130,2026-05-31,NaN
1,blinkit,abohar,718434,PADVJAS-R,2026-05-31,0.3000,2026-05-31,NaN
2,blinkit,abohar,718553,PA-BDYLOT,2026-05-31,0.4000,2026-05-31,NaN
3,blinkit,abohar,718559,SFOATS-FL,2026-05-31,0.0001,2026-05-31,NaN
4,blinkit,abohar,718681,PADV-HRCR,2026-05-31,0.1000,2026-05-31,NaN
...,...,...,...,...,...,...,...,...
656082,zepto,warangal,729186,SAFF GOLD,2025-06-30,0.0030,2025-06-30,D530
656083,zepto,warangal,729186,SAFF GOLD,2025-07-31,0.0030,2025-07-31,D530
656084,zepto,warangal,729186,SAFF GOLD,2025-08-31,0.0060,2025-08-31,D530
656085,zepto,warangal,729186,SAFF GOLD,2025-10-31,0.0030,2025-10-31,D530


In [ ]:
df1 = offtakes_monthly_df[~offtakes_monthly_df['depot'].isna()]
df2 = offtakes_monthly_df[offtakes_monthly_df['depot'].isna()]

missing_depots = pd.read_csv('/data/aman_singh/mt_forecast/depot_missing_2.csv')
missing_depots.rename(columns = {'Unnamed: 3':'depot', 'platform_name':'chain'}, inplace = True)

df2.drop(columns = ['depot'],inplace = True)
df2 = df2.merge(missing_depots[['chain', 'city', 'depot']], on = ['chain','city'], how = 'left')


In [ ]:
df2[df2['depot'].isna()]#.to_csv('missing_depots.csv')#['city'].unique()

,chain,city,parent_material_code,material_group_code,month_date,vol_in_roum,run_month,depot
0,blinkit,abohar,718398,SAFF ACTV,2026-05-31,0.0130,2026-05-31,NaN
1,blinkit,abohar,718434,PADVJAS-R,2026-05-31,0.3000,2026-05-31,NaN
2,blinkit,abohar,718553,PA-BDYLOT,2026-05-31,0.4000,2026-05-31,NaN
3,blinkit,abohar,718559,SFOATS-FL,2026-05-31,0.0001,2026-05-31,NaN
4,blinkit,abohar,718681,PADV-HRCR,2026-05-31,0.1000,2026-05-31,NaN
...,...,...,...,...,...,...,...,...
4927,zepto,-,809059,PURSNS_ML,2023-12-31,1.2000,2023-12-31,NaN
4928,zepto,-,809059,PURSNS_ML,2024-01-31,1.3500,2024-01-31,NaN
4929,zepto,-,809239,SAF_MAYO,2023-10-31,0.0253,2023-10-31,NaN
4930,zepto,-,809258,SAF_MAYO,2023-10-31,0.1197,2023-10-31,NaN


In [ ]:
df2 = df2.dropna()
offtakes_monthly_df = pd.concat([df1,df2])
offtakes_monthly_df

,chain,city,parent_material_code,material_group_code,month_date,vol_in_roum,run_month,depot
16,blinkit,agra,718312,PCNO(R),2024-08-31,0.003,2024-08-31,D115
17,blinkit,agra,718312,PCNO(R),2024-09-30,0.012,2024-09-30,D115
18,blinkit,agra,718312,PCNO(R),2024-10-31,0.024,2024-10-31,D115
19,blinkit,agra,718312,PCNO(R),2024-11-30,0.031,2024-11-30,D115
20,blinkit,agra,718312,PCNO(R),2024-12-31,0.040,2024-12-31,D115
...,...,...,...,...,...,...,...,...
9998,zepto,valsad,719008,SAFF GOLD,2026-02-28,0.003,2026-02-28,D356
9999,zepto,valsad,719008,SAFF GOLD,2026-03-31,0.009,2026-03-31,D356
10000,zepto,valsad,719008,SAFF GOLD,2026-04-30,0.015,2026-04-30,D356
10001,zepto,valsad,719008,SAFF GOLD,2026-05-31,0.036,2026-05-31,D356


In [ ]:
offtakes_monthly_df = offtakes_monthly_df.groupby(['chain', 'depot', 'parent_material_code', 'material_group_code', 'month_date'], as_index=False)[['vol_in_roum']].sum()
offtakes_monthly_df

,chain,depot,parent_material_code,material_group_code,month_date,vol_in_roum
0,blinkit,D112,718288,SAFF GOLD,2024-05-31,0.1450
1,blinkit,D112,718310,PCNO(R),2025-04-30,0.0130
2,blinkit,D112,718310,PCNO(R),2025-05-31,0.0380
3,blinkit,D112,718310,PCNO(R),2025-06-30,0.0930
4,blinkit,D112,718310,PCNO(R),2025-07-31,0.0915
...,...,...,...,...,...,...
171073,zepto,D677,810405,PURSNS_ML,2025-02-28,0.0600
171074,zepto,D677,810406,PURSNS_ML,2025-01-31,0.0300
171075,zepto,D677,810406,PURSNS_ML,2025-02-28,0.0300
171076,zepto,D677,810407,PURSNS_ML,2025-02-28,0.0300


In [ ]:
offtakes_monthly_df.duplicated(subset=['chain', 'depot','parent_material_code', 'month_date']).sum()

0

In [ ]:
offtakes_monthly_df = realign_pskus(offtakes_monthly_df.copy(), 'parent_material_code')

In [ ]:
# offtakes_monthly_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()

In [ ]:
offtakes_monthly_df = offtakes_monthly_df.groupby(
    ['chain', 'depot','parent_material_code', 'material_group_code', 
     'month_date'], as_index=False
)['vol_in_roum'].sum()

In [ ]:
offtakes_monthly_df['month_date'].max()

Timestamp('2026-05-31 00:00:00')

#### Add m month forecast

In [ ]:
mmonth_ot_df = pd.read_sql("""
SELECT * FROM TRN_DF_QCOM_OFFTAKE_CHAIN_DEPOT_PSKU
WHERE run_month='2026-07-31' AND month_date in ('2026-06-30')  
""",
dev_conn
)

In [ ]:
mmonth_ot_df

In [ ]:
mmonth_ot_df.columns = mmonth_ot_df.columns.str.lower()
mmonth_ot_df.rename(
    columns={
        'platform_name': 'chain',
        'brand_code': 'material_group_code',
        'vol_in_rum': 'vol_in_roum'
    }, 
    inplace=True
)

In [ ]:
mmonth_ot_df['parent_material_code'] = mmonth_ot_df['parent_material_code'].astype(int)

In [ ]:
mmonth_ot_df['month_date'] = pd.to_datetime(mmonth_ot_df['month_date'])
mmonth_ot_df['run_month'] = pd.to_datetime(mmonth_ot_df['run_month'])

####

In [ ]:
mmonth_ot_df.duplicated(subset= ['key','month_date','material_group_code']).sum()

In [ ]:
#qtr_ind_rate_df[['brand_code','qtr_ind_rate']]
# qtr_ind_rate_df.duplicated(subset = ['brand_code','qtr_ind_rate'],keep=False).sum()

In [ ]:
# x = mmonth_ot_df.copy()
# x.rename(columns = {'material_group_code':'Brand'},inplace = True)
# len_before_merge = len(x)
# x = x.merge(
#     qtr_ind_rate_df.drop('month_date', axis=1).rename(columns={
#         'brand_code': 'Brand',
#         'qtr_ind_rate': 'Index Rate'
#     }),
#     on=['Brand'],
#     how='left'
# )
# assert len_before_merge == len(x)
# del len_before_merge
# x.dtypes
# x['offtake_val'] = x['vol_in_roum']*x['Index Rate']/10**7
# x.groupby(['month_date'])['offtake_val'].sum().reset_index()#[20:]

In [ ]:
mmonth_ot_df

In [ ]:
# mmonth_ot_df

In [ ]:
offtakes_monthly_df = pd.concat([offtakes_monthly_df, mmonth_ot_df], ignore_index=True)

In [ ]:
offtakes_monthly_df.duplicated(subset=['chain', 'depot','parent_material_code', 'month_date']).sum()

0

In [ ]:
offtakes_monthly_df = offtakes_monthly_df[~offtakes_monthly_df['material_group_code'].isin(['LVNPST_ML'])]

In [ ]:
offtakes_monthly_df = impute_missing_dates(
    offtakes_monthly_df.copy(),
    key=['chain', 'depot','parent_material_code'],
    date_col='month_date'
)

11258it [00:02, 4593.72it/s]


In [ ]:
offtakes_monthly_df.sort_values(by=['key', 'month_date'], inplace=True)

cols = ['chain', 'depot','parent_material_code', 'material_group_code']

offtakes_monthly_df[cols] = offtakes_monthly_df.groupby('key')[cols].transform(lambda x: x.ffill().bfill())

In [ ]:
offtakes_monthly_df['run_month'] = offtakes_monthly_df['month_date'] + MonthEnd(0)

In [ ]:
offtakes_monthly_df['vol_in_roum'] = offtakes_monthly_df['vol_in_roum'].fillna(0)

In [ ]:
offtakes_monthly_df['parent_material_code'] = offtakes_monthly_df['parent_material_code'].astype(int)

In [ ]:
(offtakes_monthly_df['key'] == offtakes_monthly_df[['chain', 'depot','parent_material_code']].astype(str).agg('_'.join, axis=1)).all()

True

In [ ]:
offtakes_monthly_df.rename(columns={'vol_in_roum': 'offtake_vol_rum'}, inplace=True)

In [ ]:
offtakes_monthly_df.groupby(['month_date'])['offtake_vol_rum'].sum().reset_index()[20:]

,month_date,offtake_vol_rum
20,2024-09-30,26332.4787
21,2024-10-31,33481.2734
22,2024-11-30,50496.0371
23,2024-12-31,59153.6373
24,2025-01-31,63434.0424
25,2025-02-28,69783.5690
26,2025-03-31,77069.4825
27,2025-04-30,70404.5559
28,2025-05-31,63749.8364
29,2025-06-30,66717.0806


In [ ]:
# offtakes_monthly_df[offtakes_monthly_df['month_date'] < '2026-02-28'].to_excel('Offtakes Chain PSKU till Jan26.xlsx', index=False)

In [ ]:
offtakes_monthly_df


,month_date,key,chain,depot,parent_material_code,material_group_code,offtake_vol_rum,run_month
0,2024-05-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.145,2024-05-31
1,2024-06-30,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-06-30
2,2024-07-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-07-31
3,2024-08-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-08-31
4,2024-09-30,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-09-30
...,...,...,...,...,...,...,...,...
319690,2026-08-31,zepto_D677_810439,zepto,D677,810439,SAF-MUSLI,0.000,2026-08-31
319691,2026-09-30,zepto_D677_810439,zepto,D677,810439,SAF-MUSLI,0.000,2026-09-30
319692,2026-10-31,zepto_D677_810439,zepto,D677,810439,SAF-MUSLI,0.000,2026-10-31
319693,2026-11-30,zepto_D677_810439,zepto,D677,810439,SAF-MUSLI,0.000,2026-11-30


In [ ]:
offtakes_monthly_df['key'] = offtakes_monthly_df['key'].str.lower()

### SOH

In [ ]:
soh_query = """
SELECT 
    SOH.final_chains_ho AS chain,
    DCM.marico_depot,
    SOH.parent_sku AS parent_material_code,
    SOH.as_on_date,
    SUM(current_soh) AS current_soh
FROM dev_db.public.dwh_soh_oos SOH
LEFT JOIN 
(
    SELECT DISTINCT
        customer,
        marico_depot
    FROM
        dev_db.data_science.trn_soh_dc_master
) DCM on SOH.distributorcode = DCM.customer
WHERE 
    final_chains_ho IN ('Swiggy', 'Blinkit', 'Zepto') AND
    vol_bpm='Vol'
GROUP BY
    1, 2, 3, 4
ORDER BY
    1, 2, 3, 4
"""

soh_df = pd.read_sql(
    soh_query,
    dev_conn
)

In [ ]:
soh_df.columns = soh_df.columns.str.lower()

In [ ]:
soh_df.duplicated(subset=['chain', 'marico_depot', 'parent_material_code', 'as_on_date']).sum()

0

In [ ]:
soh_df['as_on_date'] = pd.to_datetime(soh_df['as_on_date'])
# soh_df[soh_df['as_on_date'] >= '2026-01-31'].to_csv('soh_recent_qcom.csv', index=False)

In [ ]:
soh_df = realign_pskus(soh_df.copy(), 'parent_material_code')

In [ ]:
soh_df.duplicated(subset=['chain', 'marico_depot', 'parent_material_code', 'as_on_date']).sum()

76839

In [ ]:
soh_df = soh_df.groupby(
    ['chain', 'marico_depot', 'parent_material_code', 'as_on_date'],
    as_index=False
)['current_soh'].sum()

In [ ]:
soh_df['as_on_date'].max()

Timestamp('2026-06-03 00:00:00')

In [ ]:
soh_df['current_soh'].min()

0.0

In [ ]:
soh_df.duplicated(subset=['chain', 'marico_depot', 'parent_material_code', 'as_on_date']).sum()

0

In [ ]:
soh_df['marico_depot'] = soh_df['marico_depot'].str.lower()

In [ ]:
soh_df['key2'] = soh_df[['chain', 'marico_depot', 'parent_material_code']].astype(str).agg('_'.join, axis=1)

In [ ]:
soh_df['as_on_date'] = pd.to_datetime(soh_df['as_on_date'])

In [ ]:
soh_df['run_month'] = soh_df['as_on_date'] + MonthEnd(0)

In [ ]:
date_wise_soh_vol_sum = soh_df.groupby(['as_on_date'], as_index=False)['current_soh'].sum()

In [ ]:
date_wise_soh_vol_sum[date_wise_soh_vol_sum['current_soh'] == 0]

,as_on_date,current_soh
0,2025-01-19,0.0
1,2025-01-20,0.0
19,2025-07-31,0.0
22,2025-08-03,0.0
29,2025-08-12,0.0
34,2025-08-20,0.0
40,2025-08-28,0.0
53,2025-09-15,0.0
56,2025-09-20,0.0
60,2025-09-25,0.0


In [ ]:
soh_df[
    soh_df['as_on_date'].isin(
        date_wise_soh_vol_sum[date_wise_soh_vol_sum['current_soh'] == 0]['as_on_date'].unique()
    )
]['current_soh'].sum()

0.0

In [ ]:
soh_df = soh_df[
    ~soh_df['as_on_date'].isin(
        date_wise_soh_vol_sum[date_wise_soh_vol_sum['current_soh'] == 0]['as_on_date'].unique()
    )
]

In [ ]:
soh_df['current_soh'].sum()

14535062.53978667

In [ ]:
soh_df.groupby(['chain'])['as_on_date'].max()

chain
Blinkit   2026-06-03
Swiggy    2026-06-03
Zepto     2026-06-03
Name: as_on_date, dtype: datetime64[ns]

# Remove the below cell

In [ ]:
### THIS IS TEMP FILTER ###
soh_df = soh_df[soh_df['as_on_date'] < '2026-07-01']

### Forecasts

In [ ]:
forecasts = pd.read_excel(
    r"/data/aman_singh/acuuracy_check/Heuristics_all_combination_qcom_depot_psku_july_live.xlsx",
    sheet_name='Base'
)
forecasts = forecasts[forecasts['key'].notna()]
forecasts.head()

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value,non_seasonal_growth,final_seasonal_month,final_heuristic_value,month_different,error_60_prophet,abs_error_60_prophet
0,blinkit_D112_718288,2026-03-31,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,1.0,0,0.900645,0.0,0.0,0.0
1,blinkit_D112_718288,2026-04-30,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,1.0,0,0.943984,0.0,0.0,0.0
2,blinkit_D112_718288,2026-05-31,0.0,0.0,0.034707,0.016917,0.0,0.0,0.000482,0.000235,...,0.0,0.0,0.0,0.0,1.0,0,0.910005,0.0,0.0,0.0
3,blinkit_D112_718288,2026-06-30,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,1.0,0,0.886910,0.0,0.0,0.0
4,blinkit_D112_718288,2026-07-31,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,1.0,0,0.959171,0.0,0.0,0.0


In [ ]:
xx = forecasts.copy()

In [ ]:
# forecasts = xx.copy()

In [ ]:
# forecasts.drop(['Unnamed: 0.1', 'Unnamed: 0'], axis=1, inplace=True)
forecasts['Final Heuristic Prophet Vol'] = forecasts['final_heuristic_value'] * (10 ** 7) / forecasts['qtr_ind_rate']

In [ ]:
forecasts.columns.to_list()

['key',
 'month_date',
 'pred_p3m',
 'pred_p6m',
 'pred_prophet',
 'pred_rf',
 'pred_value_p3m',
 'pred_value_p6m',
 'pred_value_prophet',
 'pred_value_rf',
 'parent_material_code',
 'platform_name',
 'brand_code',
 'qtr_ind_rate',
 'vol_in_rum_value',
 'pred_best_model',
 'pred_value_best_model',
 'vol_in_rum_treated',
 'vol_in_rum_value_treated',
 'train_till',
 'cov',
 'run',
 'step',
 'file_path',
 'run_month',
 'M month',
 'portfolio',
 'pred_prophet_70%ile',
 'vol_in_rum',
 'P3M',
 'P6M',
 'LY P3M',
 'LY P6M',
 'LY P3M_copy',
 'P3M Max',
 'P3M Top 2 Mean',
 'MoM P3M growth',
 'MoM P3M growth_lag_1',
 'MoM P3M growth_lag_2',
 '>=20%_3M_inc_month_count',
 'Avg(P3M Mean, Max)',
 'P3M_value',
 'P6M_value',
 'LY P3M_value',
 'LY P6M_value',
 'pred_prophet_70%ile_value',
 'LY',
 'LLY',
 'LY value',
 'LLY value',
 'OT_Value_in_Cr_lag_1',
 'OT_Value_in_Cr_lag_2',
 'OT_Value_in_Cr_lag_3',
 'class',
 'skipped',
 'seasonality_flag',
 'final_trend',
 'lower_threshold',
 'upper_threshold',
 '

In [ ]:
forecasts['parent_material_code'] = forecasts['parent_material_code'].astype(int)

In [ ]:
forecasts['depot'] = forecasts['key'].str.split('_').str[1]

In [ ]:
forecasts

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value,non_seasonal_growth,final_seasonal_month,final_heuristic_value,month_different,error_60_prophet,abs_error_60_prophet,Final Heuristic Prophet Vol,depot
0,blinkit_D112_718288,2026-03-31,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,...,0.0,0.0,1.0,0,0.900645,0.0,0.0,0.0,0.0,D112
1,blinkit_D112_718288,2026-04-30,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,...,0.0,0.0,1.0,0,0.943984,0.0,0.0,0.0,0.0,D112
2,blinkit_D112_718288,2026-05-31,0.0,0.0,0.034707,0.016917,0.0,0.0,0.000482,0.000235,...,0.0,0.0,1.0,0,0.910005,0.0,0.0,0.0,0.0,D112
3,blinkit_D112_718288,2026-06-30,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,...,0.0,0.0,1.0,0,0.886910,0.0,0.0,0.0,0.0,D112
4,blinkit_D112_718288,2026-07-31,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,...,0.0,0.0,1.0,0,0.959171,0.0,0.0,0.0,0.0,D112
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
273638,zepto_D674_811181,2026-05-31,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,...,NaN,NaN,NaN,0,NaN,NaN,0.0,0.0,0.0,D674
273639,zepto_D674_811181,2026-06-30,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,...,NaN,NaN,NaN,0,NaN,NaN,0.0,0.0,0.0,D674
273640,zepto_D674_811181,2026-07-31,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,...,NaN,NaN,NaN,0,NaN,NaN,0.0,0.0,0.0,D674
273641,zepto_D674_811181,2026-08-31,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,...,NaN,NaN,NaN,0,NaN,NaN,0.0,0.0,0.0,D674


In [ ]:
forecasts['key'] = forecasts[['platform_name', 'depot','parent_material_code']].astype(str).agg('_'.join, axis=1)

In [ ]:
# forecasts['Final Heuristic Prophet Vol'] = np.where(
#     forecasts['skipped'] == 1,
#     np.where(
#         (forecasts['Potential Seasonal Brand'] == 1) &
#         (forecasts['LY'].notna()),
#         forecasts['LY'],
#         forecasts['P3M']
#     ),
#     forecasts['Final Heuristic Prophet Vol']
# )

In [ ]:
# forecasts[forecasts['Final Heuristic Prophet Vol'] != forecasts['final_heuristic_60_prophet_vol_2']][['Final Heuristic Prophet Vol', 'final_heuristic_60_prophet_vol_2']]

In [ ]:
forecasts.rename(columns = {'run_month_x': 'run_month'}, inplace=True)

In [ ]:
forecasts = forecasts[['key', 'month_date', 'platform_name', 'depot','parent_material_code', 'brand_code', 
                       'run_month', 'M month', 'Final Heuristic Prophet Vol', 'skipped']]

In [ ]:
forecasts['run_month'].unique()

<DatetimeArray>
['2026-03-31 00:00:00', '2026-04-30 00:00:00', '2026-01-31 00:00:00',
 '2026-02-28 00:00:00']
Length: 4, dtype: datetime64[ns]

In [ ]:
# forecasts = forecasts[forecasts['run_month'].isin(
#     ['2025-08-31', '2025-09-30']
# )]

In [ ]:
forecasts

,key,month_date,platform_name,depot,parent_material_code,brand_code,run_month,M month,Final Heuristic Prophet Vol,skipped
0,blinkit_D112_718288,2026-03-31,blinkit,D112,718288,SAFF GOLD,2026-03-31,M,0.0,0
1,blinkit_D112_718288,2026-04-30,blinkit,D112,718288,SAFF GOLD,2026-03-31,M+1,0.0,0
2,blinkit_D112_718288,2026-05-31,blinkit,D112,718288,SAFF GOLD,2026-03-31,M+2,0.0,0
3,blinkit_D112_718288,2026-06-30,blinkit,D112,718288,SAFF GOLD,2026-03-31,M+3,0.0,0
4,blinkit_D112_718288,2026-07-31,blinkit,D112,718288,SAFF GOLD,2026-03-31,M+4,0.0,0
...,...,...,...,...,...,...,...,...,...,...
273638,zepto_D674_811181,2026-05-31,zepto,D674,811181,SAF_CDPRS,2026-02-28,M+3,0.0,1
273639,zepto_D674_811181,2026-06-30,zepto,D674,811181,SAF_CDPRS,2026-02-28,M+4,0.0,1
273640,zepto_D674_811181,2026-07-31,zepto,D674,811181,SAF_CDPRS,2026-02-28,M+5,0.0,1
273641,zepto_D674_811181,2026-08-31,zepto,D674,811181,SAF_CDPRS,2026-02-28,M+6,0.0,1


In [ ]:
forecasts.duplicated(['key', 'run_month', 'month_date']).sum()

0

In [ ]:
forecasts['key'] = forecasts['key'].str.lower()

In [ ]:
# forecasts.rename(columns={'final_vol': 'Final Heuristic Prophet Vol'}, inplace=True)

### Collate Everything

In [ ]:
print(f"SOH", soh_df['chain'].unique())
print("Offtakes:", offtakes_monthly_df['chain'].unique())
print("Plan Actuals:", plan_actuals_df['chain'].unique())

SOH ['Blinkit' 'Swiggy' 'Zepto']
Offtakes: ['blinkit' 'swiggy' 'zepto']
Plan Actuals: ['Blinkit' 'Swiggy' 'Zepto']


In [ ]:
plan_actuals_df

,month_date,key,chain,marico_depot,parent_material_code,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,Primary P3M
0,2024-01-31,Blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.001866,0.000,0.0,NaN
1,2024-02-29,Blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.003372,0.001,0.0,0.0
2,2024-03-31,Blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.002172,0.000,0.0,0.0
3,2024-04-30,Blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.004476,0.000,0.0,0.0
4,2024-05-31,Blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.003584,0.000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
581056,2026-08-31,Zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0,0.0
581057,2026-09-30,Zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0,0.0
581058,2026-10-31,Zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0,0.0
581059,2026-11-30,Zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0,0.0


In [ ]:
plan_actuals_df.rename(columns={'key': 'key2'}, inplace=True)

In [ ]:
plan_actuals_df['key2'] = plan_actuals_df['key2'].str.lower()

In [ ]:
final_df = plan_actuals_df[
    ['key2', 'chain', 'marico_depot', 'parent_material_code', 'material_group_code']
].drop_duplicates()

tmp_df = pd.DataFrame()

for rm in run_months_list: 
    mth_dates = [pd.to_datetime(rm) + MonthEnd(i) for i in range(-1, 9)]
    for mth_dt in mth_dates:
        tmp_df2 = final_df.copy()
        tmp_df2['run_month'] = pd.to_datetime(rm)
        tmp_df2['month_date'] = pd.to_datetime(mth_dt)

        tmp_df = pd.concat([tmp_df, tmp_df2], ignore_index=True)
        del tmp_df2

final_df = tmp_df.copy()    
del tmp_df

In [ ]:
final_df['key2'] = final_df['key2'].str.lower()

### Merge Plan

In [ ]:
plan_actuals_df.duplicated(subset=['key2', 'month_date']).sum()

0

In [ ]:
plan_actuals_df.head()

,month_date,key2,chain,marico_depot,parent_material_code,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,Primary P3M
0,2024-01-31,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.001866,0.000,0.0,NaN
1,2024-02-29,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.003372,0.001,0.0,0.0
2,2024-03-31,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.002172,0.000,0.0,0.0
3,2024-04-30,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.004476,0.000,0.0,0.0
4,2024-05-31,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.003584,0.000,0.0,0.0


In [ ]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    plan_actuals_df[['key2', 'month_date', 'pri_actuals_vol_rum', 'sec_apo_plan_vol_rum', 'Primary P3M']],
    on=['key2', 'month_date'],
    how='left'
)
assert len(final_df) == len_before_merge
del len_before_merge

In [ ]:
plan_actuals_df['month_date'].max()

Timestamp('2026-12-31 00:00:00')

In [ ]:
final_df

,key2,chain,marico_depot,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M
0,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2025-12-31,0.0,0.0,0.0
1,blinkit_d112_718288,Blinkit,d112,718288,SAFF GOLD,2026-01-31,2025-12-31,0.0,0.0,0.0
2,blinkit_d112_718297,Blinkit,d112,718297,PCNO(R),2026-01-31,2025-12-31,0.0,0.0,0.0
3,blinkit_d112_718299,Blinkit,d112,718299,PCNO(R),2026-01-31,2025-12-31,0.0,0.0,0.0
4,blinkit_d112_718308,Blinkit,d112,718308,PCNO(R),2026-01-31,2025-12-31,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
893075,zepto_nan_811005,Zepto,NaN,811005,PA_ESS_HO,2026-04-30,2026-12-31,0.0,0.0,0.0
893076,zepto_nan_811267,Zepto,NaN,811267,PA_ESS_HO,2026-04-30,2026-12-31,0.0,0.0,0.0
893077,zepto_nan_811269,Zepto,NaN,811269,PA_ESS_HO,2026-04-30,2026-12-31,0.0,0.0,0.0
893078,zepto_nan_811279,Zepto,NaN,811279,SAF_CDPRS,2026-04-30,2026-12-31,0.0,0.0,0.0


In [ ]:
final_df[['run_month', 'month_date']].drop_duplicates()

,run_month,month_date
0,2026-01-31,2025-12-31
22327,2026-01-31,2026-01-31
44654,2026-01-31,2026-02-28
66981,2026-01-31,2026-03-31
89308,2026-01-31,2026-04-30
111635,2026-01-31,2026-05-31
133962,2026-01-31,2026-06-30
156289,2026-01-31,2026-07-31
178616,2026-01-31,2026-08-31
200943,2026-01-31,2026-09-30


In [ ]:
mmonth_df = final_df[
    final_df['month_date'] < final_df['run_month']
]

final_df = final_df[
    final_df['month_date'] >= final_df['run_month']
]

final_df[['run_month', 'month_date']].drop_duplicates()

,run_month,month_date
22327,2026-01-31,2026-01-31
44654,2026-01-31,2026-02-28
66981,2026-01-31,2026-03-31
89308,2026-01-31,2026-04-30
111635,2026-01-31,2026-05-31
133962,2026-01-31,2026-06-30
156289,2026-01-31,2026-07-31
178616,2026-01-31,2026-08-31
200943,2026-01-31,2026-09-30
245597,2026-02-28,2026-02-28


In [ ]:
mmonth_df[['run_month', 'month_date']].drop_duplicates()

,run_month,month_date
0,2026-01-31,2025-12-31
223270,2026-02-28,2026-01-31
446540,2026-03-31,2026-02-28
669810,2026-04-30,2026-03-31


In [ ]:
final_df.sort_values(by=['run_month', 'key2', 'month_date'], inplace=True)

In [ ]:
final_df

,key2,chain,marico_depot,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M
22327,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-01-31,0.0,0.0,0.0
44654,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-02-28,0.0,0.0,0.0
66981,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-03-31,0.0,0.0,0.0
89308,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-04-30,0.0,0.0,0.0
111635,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-05-31,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
803771,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-08-31,0.0,0.0,0.0
826098,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-09-30,0.0,0.0,0.0
848425,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-10-31,0.0,0.0,0.0
870752,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-11-30,0.0,0.0,0.0


In [ ]:
for col in ['Primary P3M']:
    # if not 'LY' in col:  'LY P6M',
    final_df.loc[final_df['month_date'] > final_df['run_month'], [col]] = np.nan
    final_df[col] = final_df.groupby(['run_month', 'key2'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [ ]:
final_df = pd.concat([mmonth_df, final_df], ignore_index=True)

### Merge offtakes

In [ ]:
final_df['key'] = final_df[['chain', 'marico_depot','parent_material_code']].astype(str).agg('_'.join, axis=1)

In [ ]:
final_df['key'] = final_df['key'].str.lower()

In [ ]:
final_df.duplicated(subset=['run_month', 'month_date', 'key2']).sum()

0

In [ ]:
offtakes_monthly_df

,month_date,key,chain,depot,parent_material_code,material_group_code,offtake_vol_rum,run_month
0,2024-05-31,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.145,2024-05-31
1,2024-06-30,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-06-30
2,2024-07-31,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-07-31
3,2024-08-31,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-08-31
4,2024-09-30,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-09-30
...,...,...,...,...,...,...,...,...
319690,2026-08-31,zepto_d677_810439,zepto,D677,810439,SAF-MUSLI,0.000,2026-08-31
319691,2026-09-30,zepto_d677_810439,zepto,D677,810439,SAF-MUSLI,0.000,2026-09-30
319692,2026-10-31,zepto_d677_810439,zepto,D677,810439,SAF-MUSLI,0.000,2026-10-31
319693,2026-11-30,zepto_d677_810439,zepto,D677,810439,SAF-MUSLI,0.000,2026-11-30


In [ ]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate

qtr_ind_rate_df = read_qtr_ind_rate_table()
qtr_ind_rate_df.head()

,month_date,brand_code,qtr_ind_rate
0,2027-03-31,PA_CN_HGO,488.152
1,2027-03-31,TRU_RAWDF,800.000
2,2027-03-31,TRU_PDRFR,850.570
3,2027-03-31,TRU_OATS,177.070
4,2027-03-31,TRU_QUINO,204.750


In [ ]:
x = offtakes_monthly_df.copy()
x.rename(columns = {'material_group_code':'Brand'},inplace = True)

In [ ]:
len_before_merge = len(offtakes_monthly_df)
x = x.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(columns={
        'brand_code': 'Brand',
        'qtr_ind_rate': 'Index Rate'
    }),
    on=['Brand'],
    how='left'
)
assert len_before_merge == len(x)
del len_before_merge

In [ ]:
x.dtypes

month_date              datetime64[ns]
key                             object
chain                           object
depot                           object
parent_material_code             int64
Brand                           object
offtake_vol_rum                float64
run_month               datetime64[ns]
Index Rate                     float64
dtype: object

In [ ]:
x['offtake_val'] = x['offtake_vol_rum']*x['Index Rate']/10**7
x

,month_date,key,chain,depot,parent_material_code,Brand,offtake_vol_rum,run_month,Index Rate,offtake_val
0,2024-05-31,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.145,2024-05-31,138865.260689,0.002014
1,2024-06-30,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-06-30,138865.260689,0.000000
2,2024-07-31,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-07-31,138865.260689,0.000000
3,2024-08-31,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-08-31,138865.260689,0.000000
4,2024-09-30,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-09-30,138865.260689,0.000000
...,...,...,...,...,...,...,...,...,...,...
319690,2026-08-31,zepto_d677_810439,zepto,D677,810439,SAF-MUSLI,0.000,2026-08-31,315513.490535,0.000000
319691,2026-09-30,zepto_d677_810439,zepto,D677,810439,SAF-MUSLI,0.000,2026-09-30,315513.490535,0.000000
319692,2026-10-31,zepto_d677_810439,zepto,D677,810439,SAF-MUSLI,0.000,2026-10-31,315513.490535,0.000000
319693,2026-11-30,zepto_d677_810439,zepto,D677,810439,SAF-MUSLI,0.000,2026-11-30,315513.490535,0.000000


In [ ]:
x.groupby(['month_date'])['offtake_val'].sum().reset_index()[20:]

,month_date,offtake_val
20,2024-09-30,12.845645
21,2024-10-31,15.108964
22,2024-11-30,14.775352
23,2024-12-31,17.575673
24,2025-01-31,18.242274
25,2025-02-28,18.435689
26,2025-03-31,23.511653
27,2025-04-30,19.771691
28,2025-05-31,20.387953
29,2025-06-30,21.640044


In [ ]:
### Monthly Actual Offtakes
len_before_merge = len(final_df)
final_df = final_df.merge(
    offtakes_monthly_df[['month_date', 'key', 'offtake_vol_rum']].rename(columns={
        'offtake_vol_rum': 'Offtake Chain depot PSKU'
    }),
    on=['month_date', 'key'],
    how='left'
)   
assert len_before_merge == len(final_df)
del len_before_merge

In [ ]:
final_df

,key2,chain,marico_depot,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,key,Offtake Chain depot PSKU
0,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2025-12-31,0.0,0.0,0.0,blinkit_d112_718287,NaN
1,blinkit_d112_718288,Blinkit,d112,718288,SAFF GOLD,2026-01-31,2025-12-31,0.0,0.0,0.0,blinkit_d112_718288,0.0
2,blinkit_d112_718297,Blinkit,d112,718297,PCNO(R),2026-01-31,2025-12-31,0.0,0.0,0.0,blinkit_d112_718297,NaN
3,blinkit_d112_718299,Blinkit,d112,718299,PCNO(R),2026-01-31,2025-12-31,0.0,0.0,0.0,blinkit_d112_718299,NaN
4,blinkit_d112_718308,Blinkit,d112,718308,PCNO(R),2026-01-31,2025-12-31,0.0,0.0,0.0,blinkit_d112_718308,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
893075,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-08-31,0.0,0.0,NaN,zepto_nan_811287,NaN
893076,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-09-30,0.0,0.0,NaN,zepto_nan_811287,NaN
893077,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-10-31,0.0,0.0,NaN,zepto_nan_811287,NaN
893078,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-11-30,0.0,0.0,NaN,zepto_nan_811287,NaN


In [ ]:
# final_df['Primary Actuals Chain PSKU P3M Sum Vol'] = final_df.groupby(
#     ['run_month', 'chain', 'parent_material_code', 'month_date'], as_index=False, group_keys=False
# )['Primary P3M'].transform('sum')

In [ ]:
# final_df['Group Size'] = final_df.groupby(
#     ['run_month', 'chain', 'parent_material_code', 'month_date'], as_index=False, group_keys=False
# )['Primary P3M'].transform('size')

In [ ]:
# final_df['Fallback Contribution'] = 1 / final_df['Group Size']

In [ ]:
# final_df['Contribution'] = final_df['Primary P3M'].fillna(0) / final_df['Primary Actuals Chain PSKU P3M Sum Vol']

In [ ]:
# final_df['Final Contribution'] = np.where(
#     final_df['Contribution'].isna(),
#     final_df['Fallback Contribution'],
#     final_df['Contribution']
# )

In [ ]:
# final_df.groupby(
#     ['key', 'run_month', 'month_date']
# )['Final Contribution'].sum().max()

In [ ]:
# final_df['Offtake Chain FC PSKU'] = final_df['Final Contribution'] * final_df['Offtake Chain PSKU']

In [ ]:
mappings = {}

for run_month in final_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   

{Timestamp('2026-01-31 00:00:00'): {Timestamp('2026-01-31 00:00:00'): 'M',
  Timestamp('2026-02-28 00:00:00'): 'M+1',
  Timestamp('2026-03-31 00:00:00'): 'M+2',
  Timestamp('2026-04-30 00:00:00'): 'M+3',
  Timestamp('2026-05-31 00:00:00'): 'M+4',
  Timestamp('2026-06-30 00:00:00'): 'M+5',
  Timestamp('2026-07-31 00:00:00'): 'M+6',
  Timestamp('2026-08-31 00:00:00'): 'M+7',
  Timestamp('2026-09-30 00:00:00'): 'M+8'},
 Timestamp('2026-02-28 00:00:00'): {Timestamp('2026-02-28 00:00:00'): 'M',
  Timestamp('2026-03-31 00:00:00'): 'M+1',
  Timestamp('2026-04-30 00:00:00'): 'M+2',
  Timestamp('2026-05-31 00:00:00'): 'M+3',
  Timestamp('2026-06-30 00:00:00'): 'M+4',
  Timestamp('2026-07-31 00:00:00'): 'M+5',
  Timestamp('2026-08-31 00:00:00'): 'M+6',
  Timestamp('2026-09-30 00:00:00'): 'M+7',
  Timestamp('2026-10-31 00:00:00'): 'M+8'},
 Timestamp('2026-03-31 00:00:00'): {Timestamp('2026-03-31 00:00:00'): 'M',
  Timestamp('2026-04-30 00:00:00'): 'M+1',
  Timestamp('2026-05-31 00:00:00'): 'M+2',

In [ ]:
final_df['M month'] = final_df.apply(
    lambda x: mappings[x['run_month']].get(x['month_date'], np.nan),
    axis=1
)

In [ ]:
final_df[['run_month', 'month_date', 'M month']].drop_duplicates()

,run_month,month_date,M month
0,2026-01-31,2025-12-31,NaN
22327,2026-02-28,2026-01-31,NaN
44654,2026-03-31,2026-02-28,NaN
66981,2026-04-30,2026-03-31,NaN
89308,2026-01-31,2026-01-31,M
89309,2026-01-31,2026-02-28,M+1
89310,2026-01-31,2026-03-31,M+2
89311,2026-01-31,2026-04-30,M+3
89312,2026-01-31,2026-05-31,M+4
89313,2026-01-31,2026-06-30,M+5


### Merge Forecasts

In [ ]:
forecasts.head()

,key,month_date,platform_name,depot,parent_material_code,brand_code,run_month,M month,Final Heuristic Prophet Vol,skipped
0,blinkit_d112_718288,2026-03-31,blinkit,D112,718288,SAFF GOLD,2026-03-31,M,0.0,0
1,blinkit_d112_718288,2026-04-30,blinkit,D112,718288,SAFF GOLD,2026-03-31,M+1,0.0,0
2,blinkit_d112_718288,2026-05-31,blinkit,D112,718288,SAFF GOLD,2026-03-31,M+2,0.0,0
3,blinkit_d112_718288,2026-06-30,blinkit,D112,718288,SAFF GOLD,2026-03-31,M+3,0.0,0
4,blinkit_d112_718288,2026-07-31,blinkit,D112,718288,SAFF GOLD,2026-03-31,M+4,0.0,0


In [ ]:
forecasts.duplicated(subset=['key', 'month_date', 'run_month']).sum()

0

In [ ]:
forecasts.groupby(['month_date'])['Final Heuristic Prophet Vol'].sum().reset_index()

,month_date,Final Heuristic Prophet Vol
0,2026-01-31,103223.922380
1,2026-02-28,196633.337311
2,2026-03-31,351851.598187
3,2026-04-30,483384.876813
4,2026-05-31,466642.222142
5,2026-06-30,398450.157192
6,2026-07-31,317066.040602
7,2026-08-31,201538.972702
8,2026-09-30,74696.118842
9,2026-10-31,34188.591441


In [ ]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    forecasts[['key', 'month_date', 'run_month', 'Final Heuristic Prophet Vol']],
    on=['key', 'month_date', 'run_month'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [ ]:
final_df.rename(columns={'Final Heuristic Prophet Vol': 'Offtake Chain depot PSKU Forecast Vol'}, inplace=True)

In [ ]:
final_df.groupby(['month_date'])['Offtake Chain depot PSKU Forecast Vol'].sum().reset_index()

,month_date,Offtake Chain depot PSKU Forecast Vol
0,2025-12-31,0.000000
1,2026-01-31,102706.479134
2,2026-02-28,195622.034064
3,2026-03-31,349838.749092
4,2026-04-30,480555.607265
5,2026-05-31,464284.979695
6,2026-06-30,396101.130099
7,2026-07-31,314972.659027
8,2026-08-31,199921.612923
9,2026-09-30,73744.824946


In [ ]:
# final_df['Offtake Chain FC PSKU Forecast Vol'] = final_df['Final Contribution'] * final_df['Offtake Chain PSKU Forecast Vol']

### Norms

In [ ]:
norm_days = pd.read_csv(r"/data/aman_singh/acuuracy_check/Norms 202607.csv")

In [ ]:
norm_days['key2'] = norm_days['key2'].str.lower()

In [ ]:
norm_days

,key2,mean_soh_6m,run_month,avg_offtakes_vol_rum,norm_days
0,blinkit_d112_718288,0.000000,2026-01-31,0.000000,NaN
1,blinkit_d112_718310,0.037841,2026-01-31,0.001301,15.000000
2,blinkit_d112_718312,0.222750,2026-01-31,0.022304,9.986842
3,blinkit_d112_718317,0.000000,2026-01-31,0.000000,NaN
4,blinkit_d112_718318,0.000000,2026-01-31,0.000000,NaN
...,...,...,...,...,...
33049,zepto_d674_810520,0.000000,2026-04-30,0.001227,5.000000
33050,zepto_d674_810521,0.000000,2026-04-30,0.000718,5.000000
33051,zepto_d674_810673,0.000000,2026-04-30,0.006564,5.000000
33052,zepto_d674_810674,0.000000,2026-04-30,0.004410,5.000000


In [ ]:
norms = final_df.copy()

In [ ]:
norms = norms[['key2', 'run_month', 'month_date', 'Offtake Chain depot PSKU Forecast Vol']]

In [ ]:
norms['total_days_in_month'] = norms['month_date'].dt.day

In [ ]:
norm_days['run_month'] = pd.to_datetime(norm_days['run_month'])
norms['run_month'] = pd.to_datetime(norms['run_month'])


In [ ]:
len_before_merge = len(norms)
norms = norms.merge(
    norm_days[['key2', 'run_month','norm_days']],
    on=['key2','run_month'],
    how='left'
)
assert len_before_merge == len(norms)
del len_before_merge

In [ ]:
norms['norm_days'] = norms['norm_days'].fillna(5)

In [ ]:
norms

,key2,run_month,month_date,Offtake Chain depot PSKU Forecast Vol,total_days_in_month,norm_days
0,blinkit_d112_718287,2026-01-31,2025-12-31,NaN,31,5.0
1,blinkit_d112_718288,2026-01-31,2025-12-31,NaN,31,5.0
2,blinkit_d112_718297,2026-01-31,2025-12-31,NaN,31,5.0
3,blinkit_d112_718299,2026-01-31,2025-12-31,NaN,31,5.0
4,blinkit_d112_718308,2026-01-31,2025-12-31,NaN,31,5.0
...,...,...,...,...,...,...
893075,zepto_nan_811287,2026-04-30,2026-08-31,NaN,31,5.0
893076,zepto_nan_811287,2026-04-30,2026-09-30,NaN,30,5.0
893077,zepto_nan_811287,2026-04-30,2026-10-31,NaN,31,5.0
893078,zepto_nan_811287,2026-04-30,2026-11-30,NaN,30,5.0


In [ ]:
y = norms.copy()

In [ ]:
norms['safety_stock'] = norms['Offtake Chain depot PSKU Forecast Vol'] *  norms['norm_days'] / norms['total_days_in_month']

In [ ]:
norms.groupby(['month_date'])['safety_stock'].sum().reset_index()

,month_date,safety_stock
0,2025-12-31,0.000000
1,2026-01-31,32866.765522
2,2026-02-28,67430.108364
3,2026-03-31,107051.669647
4,2026-04-30,148270.321541
5,2026-05-31,136641.815601
6,2026-06-30,119518.611277
7,2026-07-31,89021.713827
8,2026-08-31,53428.793464
9,2026-09-30,17551.117108


In [ ]:
norms.isna().sum()

key2                                          0
run_month                                     0
month_date                                    0
Offtake Chain depot PSKU Forecast Vol    655146
total_days_in_month                           0
norm_days                                     0
safety_stock                             655146
dtype: int64

In [ ]:
norms

,key2,run_month,month_date,Offtake Chain depot PSKU Forecast Vol,total_days_in_month,norm_days,safety_stock
0,blinkit_d112_718287,2026-01-31,2025-12-31,NaN,31,5.0,NaN
1,blinkit_d112_718288,2026-01-31,2025-12-31,NaN,31,5.0,NaN
2,blinkit_d112_718297,2026-01-31,2025-12-31,NaN,31,5.0,NaN
3,blinkit_d112_718299,2026-01-31,2025-12-31,NaN,31,5.0,NaN
4,blinkit_d112_718308,2026-01-31,2025-12-31,NaN,31,5.0,NaN
...,...,...,...,...,...,...,...
893075,zepto_nan_811287,2026-04-30,2026-08-31,NaN,31,5.0,NaN
893076,zepto_nan_811287,2026-04-30,2026-09-30,NaN,30,5.0,NaN
893077,zepto_nan_811287,2026-04-30,2026-10-31,NaN,31,5.0,NaN
893078,zepto_nan_811287,2026-04-30,2026-11-30,NaN,30,5.0,NaN


In [ ]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    norms.drop(['Offtake Chain depot PSKU Forecast Vol', 'total_days_in_month', 'norm_days'], axis=1).rename(columns={
        'safety_stock': 'norms_soh'
    }),
    on=['key2', 'run_month', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [ ]:
# to get norm days safety stock of next month
norms['month_date'] = norms['month_date'] - MonthEnd(1)

In [ ]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    norms.drop(['Offtake Chain depot PSKU Forecast Vol', 'total_days_in_month'], axis=1),
    on=['key2', 'run_month', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [ ]:
final_df.sort_values(
    by=['run_month', 'key2', 'month_date'], inplace=True
)

In [ ]:
final_df['safety_stock'] = final_df['safety_stock'].fillna(0)

In [ ]:
final_df.sort_values(by=['run_month', 'key2', 'month_date'], inplace=True)

In [ ]:
norms

,key2,run_month,month_date,Offtake Chain depot PSKU Forecast Vol,total_days_in_month,norm_days,safety_stock
0,blinkit_d112_718287,2026-01-31,2025-11-30,NaN,31,5.0,NaN
1,blinkit_d112_718288,2026-01-31,2025-11-30,NaN,31,5.0,NaN
2,blinkit_d112_718297,2026-01-31,2025-11-30,NaN,31,5.0,NaN
3,blinkit_d112_718299,2026-01-31,2025-11-30,NaN,31,5.0,NaN
4,blinkit_d112_718308,2026-01-31,2025-11-30,NaN,31,5.0,NaN
...,...,...,...,...,...,...,...
893075,zepto_nan_811287,2026-04-30,2026-07-31,NaN,31,5.0,NaN
893076,zepto_nan_811287,2026-04-30,2026-08-31,NaN,30,5.0,NaN
893077,zepto_nan_811287,2026-04-30,2026-09-30,NaN,31,5.0,NaN
893078,zepto_nan_811287,2026-04-30,2026-10-31,NaN,30,5.0,NaN


In [ ]:
chain_wise_max_soh_dates = soh_df.groupby(
    ['run_month', 'chain'], as_index=False
)['as_on_date'].max()

chain_wise_max_soh_dates

,run_month,chain,as_on_date
0,2025-07-31,Blinkit,2025-07-30
1,2025-07-31,Swiggy,2025-07-30
2,2025-07-31,Zepto,2025-07-30
3,2025-08-31,Blinkit,2025-08-30
4,2025-08-31,Swiggy,2025-08-30
5,2025-08-31,Zepto,2025-08-30
6,2025-09-30,Blinkit,2025-09-30
7,2025-09-30,Swiggy,2025-09-30
8,2025-09-30,Zepto,2025-09-30
9,2025-10-31,Blinkit,2025-10-31


In [ ]:
chain_wise_max_soh_dates = (
    chain_wise_max_soh_dates
    .groupby('chain')
    .apply(lambda x: dict(zip(x['run_month'], x['as_on_date'])))
    .to_dict()
)

In [ ]:
last_date_soh = pd.DataFrame()

for c in chain_wise_max_soh_dates.keys():
    for rm in chain_wise_max_soh_dates[c].keys():
        last_date_soh = pd.concat([
            last_date_soh,
            soh_df[
                (soh_df['chain'] == c) &
                (soh_df['as_on_date'] == chain_wise_max_soh_dates[c][rm])
            ]
        ])

In [ ]:
last_date_soh['as_on_date'] = last_date_soh['as_on_date'] + MonthEnd(0)

In [ ]:
final_df['run_month'].unique()

<DatetimeArray>
['2026-01-31 00:00:00', '2026-02-28 00:00:00', '2026-03-31 00:00:00',
 '2026-04-30 00:00:00']
Length: 4, dtype: datetime64[ns]

In [ ]:
# last_date_soh = last_date_soh[last_date_soh['as_on_date']<='2026-02-01']

In [ ]:
last_date_soh

,chain,marico_depot,parent_material_code,as_on_date,current_soh,key2,run_month
52,Blinkit,d112,718288,2025-07-31,0.000,Blinkit_d112_718288,2025-07-31
322,Blinkit,d112,718310,2025-07-31,0.127,Blinkit_d112_718310,2025-07-31
592,Blinkit,d112,718312,2025-07-31,0.070,Blinkit_d112_718312,2025-07-31
860,Blinkit,d112,718317,2025-07-31,0.000,Blinkit_d112_718317,2025-07-31
906,Blinkit,d112,718318,2025-07-31,0.000,Blinkit_d112_718318,2025-07-31
...,...,...,...,...,...,...,...
2112963,Zepto,d674,810674,2026-05-31,0.000,Zepto_d674_810674,2026-05-31
2113187,Zepto,d674,810685,2026-05-31,0.000,Zepto_d674_810685,2026-05-31
2113221,Zepto,d674,810738,2026-05-31,0.000,Zepto_d674_810738,2026-05-31
2113255,Zepto,d674,811005,2026-05-31,0.000,Zepto_d674_811005,2026-05-31


In [ ]:
last_date_soh['key2'] = last_date_soh['key2'].str.lower()

In [ ]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    last_date_soh[['key2', 'as_on_date', 'current_soh']].rename(
        columns={
            'as_on_date': 'month_date',
            'current_soh': 'Actual Closing SOH'
        }
    ),
    on=['key2', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [ ]:
final_df.sort_values(by=['run_month', 'key2', 'month_date'], inplace=True)

In [ ]:
final_df

,key2,chain,marico_depot,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,key,Offtake Chain depot PSKU,M month,Offtake Chain depot PSKU Forecast Vol,norms_soh,norm_days,safety_stock,Actual Closing SOH
0,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2025-12-31,0.0,0.0,0.0,blinkit_d112_718287,NaN,NaN,NaN,NaN,5.0,0.0,NaN
1,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-01-31,0.0,0.0,0.0,blinkit_d112_718287,NaN,M,NaN,NaN,5.0,0.0,NaN
2,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-02-28,0.0,0.0,0.0,blinkit_d112_718287,NaN,M+1,NaN,NaN,5.0,0.0,NaN
3,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-03-31,0.0,0.0,0.0,blinkit_d112_718287,NaN,M+2,NaN,NaN,5.0,0.0,NaN
4,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-04-30,0.0,0.0,0.0,blinkit_d112_718287,NaN,M+3,NaN,NaN,5.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
893075,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-08-31,0.0,0.0,NaN,zepto_nan_811287,NaN,M+4,NaN,NaN,5.0,0.0,NaN
893076,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-09-30,0.0,0.0,NaN,zepto_nan_811287,NaN,M+5,NaN,NaN,5.0,0.0,NaN
893077,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-10-31,0.0,0.0,NaN,zepto_nan_811287,NaN,M+6,NaN,NaN,5.0,0.0,NaN
893078,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-11-30,0.0,0.0,NaN,zepto_nan_811287,NaN,M+7,NaN,NaN,5.0,0.0,NaN


In [ ]:
final_df['Actual Closing SOH_Lag_1'] = final_df.groupby(
    ['run_month', 'key2']
)['Actual Closing SOH'].shift(1)


final_df['Actual Closing SOH Lag 2'] = final_df.groupby(
    ['run_month', 'key2']
)['Actual Closing SOH'].shift(2)

In [ ]:
final_df['safety_stock'].sum()

789737.014427806

In [ ]:
final_df['safety_stock'].min()

0.0

In [ ]:
# final_df['Assumed Closing SOH'] = np.where(
#     final_df['M month'] == 'M',  
#     final_df['Actual Closing SOH_Lag_1'].fillna(0) + final_df['sec_apo_plan_vol_rum'].fillna(0) \
#     - final_df['Offtake Chain FC PSKU Forecast Vol'].fillna(0),
#     final_df['safety_stock']
# )

final_df['Assumed Closing SOH'] = final_df['safety_stock']

In [ ]:
final_df['Assumed Closing SOH'] = final_df['Assumed Closing SOH'].clip(lower=0.0)

In [ ]:
final_df['Assumed Closing SOH_Lag_1'] = final_df.groupby(
    ['run_month', 'key2']
)['Assumed Closing SOH'].shift(1)

final_df['Assumed Closing SOH Lag 2'] = final_df.groupby(
    ['run_month', 'key2']
)['Assumed Closing SOH'].shift(2)

In [ ]:
final_df = final_df[
    ~final_df['material_group_code'].isin(['NC FREE', 'HC FREE'])
]

In [ ]:
soh_df['as_on_date'].max()

Timestamp('2026-05-30 00:00:00')

### Add P3M, LY

In [ ]:
actuals_df = plan_actuals_df.copy()
actuals_df['month_date'] = actuals_df['month_date'] + MonthEnd(0)

In [ ]:
actuals_df = actuals_df.groupby(
    ['key2', 'month_date'], as_index=False
)[['pri_actuals_vol_rum', 'sec_actuals_vol_rum']].sum()

In [ ]:
actuals_df.duplicated(subset=['key2', 'month_date']).sum()

0

In [ ]:
actuals_df[actuals_df['month_date'] == '2026-04-30']['pri_actuals_vol_rum'].sum()

137468.246

In [ ]:
actuals_df

,key2,month_date,pri_actuals_vol_rum,sec_actuals_vol_rum
0,blinkit_d112_718287,2024-01-31,0.0,0.0
1,blinkit_d112_718287,2024-02-29,0.0,0.0
2,blinkit_d112_718287,2024-03-31,0.0,0.0
3,blinkit_d112_718287,2024-04-30,0.0,0.0
4,blinkit_d112_718287,2024-05-31,0.0,0.0
...,...,...,...,...
581056,zepto_nan_811287,2026-08-31,0.0,0.0
581057,zepto_nan_811287,2026-09-30,0.0,0.0
581058,zepto_nan_811287,2026-10-31,0.0,0.0
581059,zepto_nan_811287,2026-11-30,0.0,0.0


In [ ]:
actuals_df['month_date'].min()

Timestamp('2024-01-31 00:00:00')

In [ ]:
actuals_df.sort_values(by=['key2', 'month_date'], inplace=True)

In [ ]:
actuals_df['Primary P3M redundant'] = actuals_df.groupby(
['key2'], as_index = False, group_keys = False)['pri_actuals_vol_rum'].shift(1)\
                            .rolling(window=3, min_periods=3).mean()

In [ ]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    actuals_df.rename(columns={
        'pri_actuals_vol_rum': 'Primary Actuals Vol',
        'sec_actuals_vol_rum': 'Sec Actuals Vol'
    }),
    on=['key2', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [ ]:
final_df.isna().sum()

key2                                          0
chain                                         0
marico_depot                              41080
parent_material_code                          0
material_group_code                           0
run_month                                     0
month_date                                    0
pri_actuals_vol_rum                       33995
sec_apo_plan_vol_rum                      33995
Primary P3M                               96482
key                                           0
Offtake Chain depot PSKU                 518681
M month                                   89308
Offtake Chain depot PSKU Forecast Vol    655146
norms_soh                                655146
norm_days                                 89308
safety_stock                                  0
Actual Closing SOH                       746586
Actual Closing SOH_Lag_1                 746586
Actual Closing SOH Lag 2                 746586
Assumed Closing SOH                     

In [ ]:
plan_actuals_df

,month_date,key2,chain,marico_depot,parent_material_code,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,Primary P3M
0,2024-01-31,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.001866,0.000,0.0,NaN
1,2024-02-29,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.003372,0.001,0.0,0.0
2,2024-03-31,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.002172,0.000,0.0,0.0
3,2024-04-30,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.004476,0.000,0.0,0.0
4,2024-05-31,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.003584,0.000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
581056,2026-08-31,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0,0.0
581057,2026-09-30,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0,0.0
581058,2026-10-31,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0,0.0
581059,2026-11-30,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0,0.0


In [ ]:
final_df

,key2,chain,marico_depot,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,...,safety_stock,Actual Closing SOH,Actual Closing SOH_Lag_1,Actual Closing SOH Lag 2,Assumed Closing SOH,Assumed Closing SOH_Lag_1,Assumed Closing SOH Lag 2,Primary Actuals Vol,Sec Actuals Vol,Primary P3M redundant
0,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2025-12-31,0.0,0.0,0.0,...,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0,0.0,0.0
1,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-01-31,0.0,0.0,0.0,...,0.0,NaN,NaN,NaN,0.0,0.0,NaN,0.0,0.0,0.0
2,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-02-28,0.0,0.0,0.0,...,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0
3,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-03-31,0.0,0.0,0.0,...,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0
4,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-04-30,0.0,0.0,0.0,...,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
893075,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-08-31,0.0,0.0,NaN,...,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0
893076,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-09-30,0.0,0.0,NaN,...,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0
893077,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-10-31,0.0,0.0,NaN,...,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0
893078,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-11-30,0.0,0.0,NaN,...,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
ly_actuals_df = actuals_df.copy()

In [ ]:
ly_actuals_df['month_date'] = ly_actuals_df['month_date'] + MonthEnd(12)

In [ ]:
ly_actuals_df.rename(columns={
        'pri_actuals_vol_rum': 'LY Primary Actuals Vol',
        'sec_actuals_vol_rum': 'LY Sec Actuals Vol',
        'Primary P3M redundant': 'LY Primary P3M'
    })

,key2,month_date,LY Primary Actuals Vol,LY Sec Actuals Vol,LY Primary P3M
0,blinkit_d112_718287,2025-01-31,0.0,0.0,NaN
1,blinkit_d112_718287,2025-02-28,0.0,0.0,NaN
2,blinkit_d112_718287,2025-03-31,0.0,0.0,NaN
3,blinkit_d112_718287,2025-04-30,0.0,0.0,0.0
4,blinkit_d112_718287,2025-05-31,0.0,0.0,0.0
...,...,...,...,...,...
581056,zepto_nan_811287,2027-08-31,0.0,0.0,0.0
581057,zepto_nan_811287,2027-09-30,0.0,0.0,0.0
581058,zepto_nan_811287,2027-10-31,0.0,0.0,0.0
581059,zepto_nan_811287,2027-11-30,0.0,0.0,0.0


In [ ]:
ly_actuals_df['pri_actuals_vol_rum_lag_1'] = ly_actuals_df.groupby(
    ['key2']
)['pri_actuals_vol_rum'].shift(1)

ly_actuals_df['pri_actuals_vol_rum_lag_2'] = ly_actuals_df.groupby(
    ['key2']
)['pri_actuals_vol_rum'].shift(2)

ly_actuals_df['pri_actuals_vol_rum_lag_3'] = ly_actuals_df.groupby(
    ['key2']
)['pri_actuals_vol_rum'].shift(3)


ly_actuals_df['pri_actuals_vol_rum_lead_1'] = ly_actuals_df.groupby(
    ['key2']
)['pri_actuals_vol_rum'].shift(-1)

ly_actuals_df['pri_actuals_vol_rum_lead_2'] = ly_actuals_df.groupby(
    ['key2']
)['pri_actuals_vol_rum'].shift(-2)


In [ ]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    ly_actuals_df.rename(columns={
        'pri_actuals_vol_rum': 'LY Primary Actuals Vol',
        'sec_actuals_vol_rum': 'LY Sec Actuals Vol',
        'Primary P3M redundant': 'LY Primary P3M',
        'pri_actuals_vol_rum_lag_1': 'LY Primary Actuals Lag 1 Vol',
        'pri_actuals_vol_rum_lag_2': 'LY Primary Actuals Lag 2 Vol',
        'pri_actuals_vol_rum_lag_3': 'LY Primary Actuals Lag 3 Vol',
        'pri_actuals_vol_rum_lead_1': 'LY Primary Actuals Lead 1 Vol',
        'pri_actuals_vol_rum_lead_2': 'LY Primary Actuals Lead 2 Vol',
    }),
    on=['key2', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [ ]:
# final_df[final_df['Contribution'].isna()]

In [ ]:
offtakes_monthly_df['month_date'].max()

Timestamp('2026-12-31 00:00:00')

In [ ]:
final_offtakes_historical_df = offtakes_monthly_df.copy()

In [ ]:
final_offtakes_historical_df

,month_date,key,chain,depot,parent_material_code,material_group_code,offtake_vol_rum,run_month
0,2024-05-31,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.145,2024-05-31
1,2024-06-30,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-06-30
2,2024-07-31,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-07-31
3,2024-08-31,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-08-31
4,2024-09-30,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-09-30
...,...,...,...,...,...,...,...,...
319690,2026-08-31,zepto_d677_810439,zepto,D677,810439,SAF-MUSLI,0.000,2026-08-31
319691,2026-09-30,zepto_d677_810439,zepto,D677,810439,SAF-MUSLI,0.000,2026-09-30
319692,2026-10-31,zepto_d677_810439,zepto,D677,810439,SAF-MUSLI,0.000,2026-10-31
319693,2026-11-30,zepto_d677_810439,zepto,D677,810439,SAF-MUSLI,0.000,2026-11-30


In [ ]:
final_offtakes_historical_df.duplicated(subset=['chain', 'depot','parent_material_code', 'month_date']).sum()

0

In [ ]:
final_offtakes_historical_df['key'] = final_offtakes_historical_df[['chain', 'depot','parent_material_code']].astype(str).agg(
    '_'.join, axis=1
)

In [ ]:

final_offtakes_historical_df.sort_values(
    by=['key', 'month_date'], inplace=True
)

In [ ]:
final_offtakes_historical_df['P3M'] = final_offtakes_historical_df.groupby(
    ['key'], as_index = False, group_keys = False)['offtake_vol_rum'].shift(1)\
                                .rolling(window=3, min_periods=3).mean()

In [ ]:
final_offtakes_historical_df['key'] = final_offtakes_historical_df['key'].str.lower()

In [ ]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    final_offtakes_historical_df[['key', 'month_date', 'offtake_vol_rum', 'P3M']].rename(
        columns={'P3M': 'Offtake P3M', 'offtake_vol_rum': 'Offtake Actuals Vol'}
    ),
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [ ]:
ly_final_offtakes_historical_df = final_offtakes_historical_df.copy()

In [ ]:
ly_final_offtakes_historical_df['month_date'].min()

Timestamp('2023-01-31 00:00:00')

In [ ]:
ly_final_offtakes_historical_df['month_date'] = ly_final_offtakes_historical_df['month_date'] + MonthEnd(12)
ly_final_offtakes_historical_df.head()

,month_date,key,chain,depot,parent_material_code,material_group_code,offtake_vol_rum,run_month,P3M
0,2025-05-31,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.145,2024-05-31,NaN
1,2025-06-30,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-06-30,NaN
2,2025-07-31,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-07-31,NaN
3,2025-08-31,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-08-31,0.048333
4,2025-09-30,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-09-30,0.000000


In [ ]:
ly_final_offtakes_historical_df['LY Offtake Actuals Lag 1 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(1)

ly_final_offtakes_historical_df['LY Offtake Actuals Lag 2 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(2)

ly_final_offtakes_historical_df['LY Offtake Actuals Lag 3 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(3)


In [ ]:
ly_final_offtakes_historical_df['LY Offtake Actuals Lead 1 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(-1)

ly_final_offtakes_historical_df['LY Offtake Actuals Lead 2 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(-2)

In [ ]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    ly_final_offtakes_historical_df[['key', 'month_date', 'offtake_vol_rum', 'P3M', 'LY Offtake Actuals Lag 1 Vol', 
                                     'LY Offtake Actuals Lag 2 Vol', 
                                     'LY Offtake Actuals Lag 3 Vol', 'LY Offtake Actuals Lead 1 Vol',
                                     'LY Offtake Actuals Lead 2 Vol']].rename(
        columns={'P3M': 'LY Offtake P3M', 'offtake_vol_rum': 'LY Offtake Actuals Vol'}
    ),
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [ ]:
final_df.sort_values(by=['run_month', 'key2', 'month_date'], inplace=True)

In [ ]:
for col in ['Primary P3M', 'LY Primary P3M', 'Offtake P3M', 'LY Offtake P3M', 'Primary P3M redundant']:
    # if not 'LY' in col:  'LY P6M',
    final_df.loc[final_df['month_date'] > final_df['run_month'], [col]] = np.nan
    final_df[col] = final_df.groupby(['run_month', 'key2'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [ ]:
final_offtakes_historical_df

,month_date,key,chain,depot,parent_material_code,material_group_code,offtake_vol_rum,run_month,P3M
0,2024-05-31,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.145,2024-05-31,NaN
1,2024-06-30,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-06-30,NaN
2,2024-07-31,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-07-31,NaN
3,2024-08-31,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-08-31,0.048333
4,2024-09-30,blinkit_d112_718288,blinkit,D112,718288,SAFF GOLD,0.000,2024-09-30,0.000000
...,...,...,...,...,...,...,...,...,...
319690,2026-08-31,zepto_d677_810439,zepto,D677,810439,SAF-MUSLI,0.000,2026-08-31,0.000000
319691,2026-09-30,zepto_d677_810439,zepto,D677,810439,SAF-MUSLI,0.000,2026-09-30,0.000000
319692,2026-10-31,zepto_d677_810439,zepto,D677,810439,SAF-MUSLI,0.000,2026-10-31,0.000000
319693,2026-11-30,zepto_d677_810439,zepto,D677,810439,SAF-MUSLI,0.000,2026-11-30,0.000000


In [ ]:
final_offtakes_historical_df.sort_values(by=['key', 'month_date'], inplace=True)

In [ ]:
# final_offtakes_historical_df['OT_Lag_1'] = final_offtakes_historical_df.groupby(
#     ['material_group_code', 'key']
# )['offtake_vol_rum'].shift(1)

# final_offtakes_historical_df['OT_Lag_2'] = final_offtakes_historical_df.groupby(
#     ['material_group_code', 'key']
# )['offtake_vol_rum'].shift(2)

# final_offtakes_historical_df['OT_Lag_3'] = final_offtakes_historical_df.groupby(
#     ['material_group_code', 'key']
# )['offtake_vol_rum'].shift(3)

In [ ]:
# del final_df['OT_Lag_1'], final_df['OT_Lag_2'], final_df['OT_Lag_3']

In [ ]:
# len_before_merge = len(final_df)
# final_df = final_df.merge(
#     final_offtakes_historical_df[['key', 'month_date', 'OT_Lag_1', 'OT_Lag_2', 'OT_Lag_3']].rename(
#         columns={'month_date': 'run_month'}
#     ),
#     on=['key', 'run_month'],
#     how='left'
# )
# assert len_before_merge == len(final_df)
# del len_before_merge

In [ ]:
lags_df = plan_actuals_df.copy()

In [ ]:
lags_df.sort_values(by=['key2', 'month_date'], inplace=True)

In [ ]:
lags_df

,month_date,key2,chain,marico_depot,parent_material_code,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,Primary P3M
0,2024-01-31,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.001866,0.000,0.0,NaN
1,2024-02-29,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.003372,0.001,0.0,0.0
2,2024-03-31,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.002172,0.000,0.0,0.0
3,2024-04-30,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.004476,0.000,0.0,0.0
4,2024-05-31,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),0.0,0.003584,0.000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
581056,2026-08-31,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0,0.0
581057,2026-09-30,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0,0.0
581058,2026-10-31,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0,0.0
581059,2026-11-30,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,0.0,0.000000,0.000,0.0,0.0


In [ ]:
lags_df['Primary_Lag_2'] = lags_df.groupby(
    ['material_group_code', 'key2']
)['pri_actuals_vol_rum'].shift(1)

lags_df['Primary_Lag_3'] = lags_df.groupby(
    ['material_group_code', 'key2']
)['pri_actuals_vol_rum'].shift(2)

In [ ]:
lags_df['month_date'] = lags_df['month_date'] + MonthEnd(1)

In [ ]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    lags_df[['key2', 'month_date', 'pri_actuals_vol_rum', 'Primary_Lag_2', 'Primary_Lag_3']].rename(columns={
        'month_date': 'run_month',
        'pri_actuals_vol_rum': 'Primary_Lag_1'
    }),
    on=['key2', 'run_month'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [ ]:
final_df

,key2,chain,marico_depot,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,...,LY Offtake Actuals Vol,LY Offtake P3M,LY Offtake Actuals Lag 1 Vol,LY Offtake Actuals Lag 2 Vol,LY Offtake Actuals Lag 3 Vol,LY Offtake Actuals Lead 1 Vol,LY Offtake Actuals Lead 2 Vol,Primary_Lag_1,Primary_Lag_2,Primary_Lag_3
0,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2025-12-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
1,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-01-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
2,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-02-28,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
3,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-03-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
4,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-04-30,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
893075,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-08-31,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
893076,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-09-30,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
893077,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-10-31,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
893078,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-11-30,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
lags_final_offtakes_historical_df = final_offtakes_historical_df.copy()

lags_final_offtakes_historical_df.sort_values(by=['key', 'month_date'], inplace=True)
lags_final_offtakes_historical_df['OT_Lag_2'] = lags_final_offtakes_historical_df.groupby(
    ['material_group_code', 'key']
)['offtake_vol_rum'].shift(1)

lags_final_offtakes_historical_df['OT_Lag_3'] = lags_final_offtakes_historical_df.groupby(
    ['material_group_code', 'key']
)['offtake_vol_rum'].shift(2)

In [ ]:
# del final_df['OT_Lag_1'], final_df['OT_Lag_2'], final_df['OT_Lag_3']

In [ ]:
lags_final_offtakes_historical_df['month_date'] = lags_final_offtakes_historical_df['month_date'] + MonthEnd(1)

In [ ]:
lags_final_offtakes_historical_df['key'] = lags_final_offtakes_historical_df['key'].str.lower()
len_before_merge = len(final_df)
final_df = final_df.merge(
    lags_final_offtakes_historical_df[['key', 'month_date', 'offtake_vol_rum', 'OT_Lag_2', 'OT_Lag_3']].rename(columns={
        'month_date': 'run_month',
        'offtake_vol_rum': 'OT_Lag_1'
    }),
    on=['key', 'run_month'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [ ]:
final_df

,key2,chain,marico_depot,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,...,LY Offtake Actuals Lag 2 Vol,LY Offtake Actuals Lag 3 Vol,LY Offtake Actuals Lead 1 Vol,LY Offtake Actuals Lead 2 Vol,Primary_Lag_1,Primary_Lag_2,Primary_Lag_3,OT_Lag_1,OT_Lag_2,OT_Lag_3
0,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2025-12-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
1,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-01-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
2,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-02-28,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
3,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-03-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
4,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-04-30,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
893075,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-08-31,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
893076,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-09-30,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
893077,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-10-31,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
893078,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-11-30,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
final_df

,key2,chain,marico_depot,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,...,LY Offtake Actuals Lag 2 Vol,LY Offtake Actuals Lag 3 Vol,LY Offtake Actuals Lead 1 Vol,LY Offtake Actuals Lead 2 Vol,Primary_Lag_1,Primary_Lag_2,Primary_Lag_3,OT_Lag_1,OT_Lag_2,OT_Lag_3
0,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2025-12-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
1,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-01-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
2,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-02-28,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
3,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-03-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
4,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-04-30,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
893075,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-08-31,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
893076,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-09-30,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
893077,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-10-31,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
893078,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-11-30,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
final_df.columns

Index(['key2', 'chain', 'marico_depot', 'parent_material_code',
       'material_group_code', 'run_month', 'month_date', 'pri_actuals_vol_rum',
       'sec_apo_plan_vol_rum', 'Primary P3M', 'key',
       'Offtake Chain depot PSKU', 'M month',
       'Offtake Chain depot PSKU Forecast Vol', 'norms_soh', 'norm_days',
       'safety_stock', 'Actual Closing SOH', 'Actual Closing SOH_Lag_1',
       'Actual Closing SOH Lag 2', 'Assumed Closing SOH',
       'Assumed Closing SOH_Lag_1', 'Assumed Closing SOH Lag 2',
       'Primary Actuals Vol', 'Sec Actuals Vol', 'Primary P3M redundant',
       'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
       'LY Primary Actuals Lead 2 Vol', 'Offtake Actuals Vol', 'Offtake P3M',
       'LY Offtake Actuals Vol', 'LY Offtake P3M',
       'LY Offtake Actuals Lag 1 Vol', 'LY Offtake Actuals Lag 2 Vol

In [ ]:
final_df['Offtake Chain depot PSKU Lag 1 Vol'] = final_df['OT_Lag_1'] 
final_df['Offtake Chain depot PSKU Lag 2 Vol'] = final_df['OT_Lag_2'] 
final_df['Offtake Chain depot PSKU Lag 3 Vol'] = final_df['OT_Lag_3'] 

final_df['Offtake Chain depot PSKU P3M Vol'] = final_df['Offtake P3M'] 
final_df['LY Offtake Chain depot PSKU P3M Vol'] = final_df['LY Offtake P3M'] 

final_df['LY Offtake Chain depot PSKU Lag 1 Vol'] = final_df['LY Offtake Actuals Lag 1 Vol'] 
final_df['LY Offtake Chain depot PSKU Lag 2 Vol'] = final_df['LY Offtake Actuals Lag 2 Vol'] 
final_df['LY Offtake Chain depot PSKU Lag 3 Vol'] = final_df['LY Offtake Actuals Lag 3 Vol'] 

final_df['LY Offtake Chain depot PSKU Lead 1 Vol'] = final_df['LY Offtake Actuals Lead 1 Vol'] 
final_df['LY Offtake Chain depot PSKU Lead 2 Vol'] = final_df['LY Offtake Actuals Lead 2 Vol'] 

In [ ]:
final_df['LY Offtake Chain depot PSKU Actuals Vol'] = final_df['LY Offtake Actuals Vol']

In [ ]:
final_df

,key2,chain,marico_depot,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,...,Offtake Chain depot PSKU Lag 2 Vol,Offtake Chain depot PSKU Lag 3 Vol,Offtake Chain depot PSKU P3M Vol,LY Offtake Chain depot PSKU P3M Vol,LY Offtake Chain depot PSKU Lag 1 Vol,LY Offtake Chain depot PSKU Lag 2 Vol,LY Offtake Chain depot PSKU Lag 3 Vol,LY Offtake Chain depot PSKU Lead 1 Vol,LY Offtake Chain depot PSKU Lead 2 Vol,LY Offtake Chain depot PSKU Actuals Vol
0,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2025-12-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-01-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-02-28,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-03-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-04-30,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
893075,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-08-31,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
893076,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-09-30,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
893077,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-10-31,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
893078,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-11-30,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
final_df.shape

(893080, 60)

### Format

In [ ]:
brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")

In [ ]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate

qtr_ind_rate_df = read_qtr_ind_rate_table()
qtr_ind_rate_df.head()

,month_date,brand_code,qtr_ind_rate
0,2027-03-31,PA_CN_HGO,488.152
1,2027-03-31,TRU_RAWDF,800.000
2,2027-03-31,TRU_PDRFR,850.570
3,2027-03-31,TRU_OATS,177.070
4,2027-03-31,TRU_QUINO,204.750


In [ ]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(
        columns={'brand_code': 'material_group_code'}
    ),
    on=['material_group_code'], 
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [ ]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    brand_md_df.rename(
        columns={'brand_code': 'material_group_code'}
    ),
    on=['material_group_code'], 
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [ ]:
final_df.columns

Index(['key2', 'chain', 'marico_depot', 'parent_material_code',
       'material_group_code', 'run_month', 'month_date', 'pri_actuals_vol_rum',
       'sec_apo_plan_vol_rum', 'Primary P3M', 'key',
       'Offtake Chain depot PSKU', 'M month',
       'Offtake Chain depot PSKU Forecast Vol', 'norms_soh', 'norm_days',
       'safety_stock', 'Actual Closing SOH', 'Actual Closing SOH_Lag_1',
       'Actual Closing SOH Lag 2', 'Assumed Closing SOH',
       'Assumed Closing SOH_Lag_1', 'Assumed Closing SOH Lag 2',
       'Primary Actuals Vol', 'Sec Actuals Vol', 'Primary P3M redundant',
       'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
       'LY Primary Actuals Lead 2 Vol', 'Offtake Actuals Vol', 'Offtake P3M',
       'LY Offtake Actuals Vol', 'LY Offtake P3M',
       'LY Offtake Actuals Lag 1 Vol', 'LY Offtake Actuals Lag 2 Vol

In [ ]:
final_df = final_df.rename(columns={
    'key2': 'Key2',
    'key': 'Key',
    'chain': 'Chain',
    'marico_depot': 'Depot',
    'parent_material_code': 'PSKU',
    'material_group_code': 'Brand',
    'run_month': 'Run Month',
    'month_date': 'Month Date',
    'pri_actuals_vol_rum': 'Primary Till Date Actuals Vol',
    'sec_apo_plan_vol_rum': 'Secondary Plan Vol',
    'Primary P3M': 'Primary P3M Vol',
    'Offtake Chain PSKU': 'Offtake Chain PSKU Vol',
    'Offtake Chain depot PSKU': 'Offtake Chain Depot PSKU Vol',
    'norms_soh': 'Norms SOH',
    'norm_days': 'Norm Days',
    'safety_stock': 'Safety Stock Vol',
    'Actual Closing SOH': 'Actual Closing SOH Vol',
    'Actual Closing SOH_Lag_1': 'Actual Closing SOH Lag 1 Vol',
    'Actual Closing SOH Lag 2': 'Actual Closing SOH Lag 2 Vol',
    'Assumed Closing SOH': 'Assumed Closing SOH Vol',
    'Assumed Closing SOH_Lag_1': 'Assumed Closing SOH Lag 1 Vol',
    'Assumed Closing SOH Lag 2': 'Assumed Closing SOH Lag 2 Vol',
    'Primary P3M redundant': 'Primary P3M redundant Vol',
    'LY Primary P3M': 'LY Primary P3M Vol',
    'Offtake P3M': 'Offtake P3M Vol',
    'LY Offtake P3M': 'LY Offtake P3M Vol',
    'Primary_Lag_1': 'Primary Actuals Lag 1 Vol',
    'Primary_Lag_2': 'Primary Actuals Lag 2 Vol',
    'Primary_Lag_3': 'Primary Actuals Lag 3 Vol',
    'OT_Lag_1': 'Offtake Actuals Lag 1 Vol',
    'OT_Lag_2': 'Offtake Actuals Lag 2 Vol',
    'OT_Lag_3': 'Offtake Actuals Lag 3 Vol',
    'qtr_ind_rate': 'Index Rate',
    'portfolio': 'Portfolio'
})

In [ ]:
final_df.columns

Index(['Key2', 'Chain', 'Depot', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Primary Till Date Actuals Vol', 'Secondary Plan Vol',
       'Primary P3M Vol', 'Key', 'Offtake Chain Depot PSKU Vol', 'M month',
       'Offtake Chain depot PSKU Forecast Vol', 'Norms SOH', 'Norm Days',
       'Safety Stock Vol', 'Actual Closing SOH Vol',
       'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol',
       'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol',
       'Assumed Closing SOH Lag 2 Vol', 'Primary Actuals Vol',
       'Sec Actuals Vol', 'Primary P3M redundant Vol',
       'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M Vol',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
       'LY Primary Actuals Lead 2 Vol', 'Offtake Actuals Vol',
       'Offtake P3M Vol', 'LY Offtake Actuals Vol', 'LY Offtake P3M Vol',
       'LY Offtake Actuals Lag 1 Vol', 'LY Of

In [ ]:
columns_order = [
    'Key2', 'Key', 'Chain', 'Depot', 'PSKU', 'Brand', 'Index Rate', 'Portfolio', 
    'Run Month', 'Month Date', 'M month', 
    
    'Primary Till Date Actuals Vol', 'Secondary Plan Vol', 'Primary P3M Vol', 

   'Offtake Chain Depot PSKU Vol', 
    'Offtake Chain depot PSKU Forecast Vol',
    
    'Norms SOH', 'Norm Days', 'Safety Stock Vol', 'Actual Closing SOH Vol',
    'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol', 'Assumed Closing SOH Vol',
    'Assumed Closing SOH Lag 1 Vol', 'Assumed Closing SOH Lag 2 Vol', 
    'Final Assumed Closing SOH Vol', 'Final Assumed Closing SOH Lag 1 Vol',
    
    'Primary Actuals Vol', 'Sec Actuals Vol', 'Primary P3M redundant Vol',
    'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M Vol',

    'Offtake Actuals Vol', 'Offtake P3M Vol', 'LY Offtake Actuals Vol', 'LY Offtake P3M Vol', 
    
    'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol', 'Primary Actuals Lag 3 Vol',
    'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol', 
    'LY Primary Actuals Lead 1 Vol', 'LY Primary Actuals Lead 2 Vol',

    'Offtake Actuals Lag 1 Vol', 'Offtake Actuals Lag 2 Vol',
    'Offtake Actuals Lag 3 Vol', 'LY Offtake Actuals Lag 1 Vol', 
    'LY Offtake Actuals Lag 2 Vol', 'LY Offtake Actuals Lag 3 Vol',
    'LY Offtake Actuals Lead 1 Vol', 'LY Offtake Actuals Lead 2 Vol',

    'Offtake Chain depot PSKU Lag 1 Vol', 'Offtake Chain depot PSKU Lag 2 Vol', 
    'Offtake Chain depot PSKU Lag 3 Vol', 'Offtake Chain depot PSKU P3M Vol', 
    'LY Offtake Chain depot PSKU P3M Vol', 'LY Offtake Chain depot PSKU Actuals Vol',
    'LY Offtake Chain depot PSKU Lag 1 Vol', 'LY Offtake Chain depot PSKU Lag 2 Vol', 
    'LY Offtake Chain depot PSKU Lag 3 Vol', 'LY Offtake Chain depot PSKU Lead 1 Vol',
    'LY Offtake Chain depot PSKU Lead 2 Vol',

    'Calculated Primary Vol'
]

In [ ]:
for col in final_df.columns:
    try:
        assert col in columns_order
    except:
        print(col)

In [ ]:
final_df.columns

Index(['Key2', 'Chain', 'Depot', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Primary Till Date Actuals Vol', 'Secondary Plan Vol',
       'Primary P3M Vol', 'Key', 'Offtake Chain Depot PSKU Vol', 'M month',
       'Offtake Chain depot PSKU Forecast Vol', 'Norms SOH', 'Norm Days',
       'Safety Stock Vol', 'Actual Closing SOH Vol',
       'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol',
       'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol',
       'Assumed Closing SOH Lag 2 Vol', 'Primary Actuals Vol',
       'Sec Actuals Vol', 'Primary P3M redundant Vol',
       'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M Vol',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
       'LY Primary Actuals Lead 2 Vol', 'Offtake Actuals Vol',
       'Offtake P3M Vol', 'LY Offtake Actuals Vol', 'LY Offtake P3M Vol',
       'LY Offtake Actuals Lag 1 Vol', 'LY Of

In [ ]:
final_df

,Key2,Chain,Depot,PSKU,Brand,Run Month,Month Date,Primary Till Date Actuals Vol,Secondary Plan Vol,Primary P3M Vol,...,Offtake Chain depot PSKU P3M Vol,LY Offtake Chain depot PSKU P3M Vol,LY Offtake Chain depot PSKU Lag 1 Vol,LY Offtake Chain depot PSKU Lag 2 Vol,LY Offtake Chain depot PSKU Lag 3 Vol,LY Offtake Chain depot PSKU Lead 1 Vol,LY Offtake Chain depot PSKU Lead 2 Vol,LY Offtake Chain depot PSKU Actuals Vol,Index Rate,Portfolio
0,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2025-12-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,349274.00142,CNO
1,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-01-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,349274.00142,CNO
2,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-02-28,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,349274.00142,CNO
3,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-03-31,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,349274.00142,CNO
4,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),2026-01-31,2026-04-30,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,349274.00142,CNO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
893075,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-08-31,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,740.00000,NaN
893076,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-09-30,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,740.00000,NaN
893077,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-10-31,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,740.00000,NaN
893078,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,2026-04-30,2026-11-30,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,740.00000,NaN


In [ ]:
def calculate_primary_iteratively(key_df):
    key_df = key_df.sort_values('Month Date').copy()
    key_df = key_df[key_df['Month Date'] >= key_df['Run Month']]

    # Fill NaNs
    fill_cols = [
        'Offtake Chain depot PSKU Forecast Vol',
        'Safety Stock Vol',
        'Actual Closing SOH Lag 1 Vol'
    ]
    key_df[fill_cols] = key_df[fill_cols].fillna(0)

    _key = key_df['Key2'].iloc[0]
    _run_month = key_df['Run Month'].iloc[0]

    outputs = []
    prev_soh = None

    for _, row in key_df.iterrows():
        forecast = row['Offtake Chain depot PSKU Forecast Vol']
        safety_stock = row['Safety Stock Vol']

        if row['M month'] == 'M':
            opening_soh = row['Actual Closing SOH Lag 1 Vol']
        else:
            opening_soh = prev_soh

        primary_vol = max(
            forecast + safety_stock - opening_soh,
            0
        )

        assumed_closing_soh = max(
            opening_soh + primary_vol - forecast,
            0
        )

        outputs.append({
            'Run Month': _run_month,
            'Key2': _key,
            'Month Date': row['Month Date'],
            'Calculated Primary Vol': primary_vol,
            'Final Assumed Closing SOH Vol': assumed_closing_soh
        })

        prev_soh = assumed_closing_soh

    return outputs

In [ ]:
tx = final_df[['Offtake Chain depot PSKU Forecast Vol',
        'Safety Stock Vol',
        'Actual Closing SOH Lag 1 Vol','Run Month','Month Date','Key2']]
tx

,Offtake Chain depot PSKU Forecast Vol,Safety Stock Vol,Actual Closing SOH Lag 1 Vol,Run Month,Month Date,Key2
0,NaN,0.0,NaN,2026-01-31,2025-12-31,blinkit_d112_718287
1,NaN,0.0,NaN,2026-01-31,2026-01-31,blinkit_d112_718287
2,NaN,0.0,NaN,2026-01-31,2026-02-28,blinkit_d112_718287
3,NaN,0.0,NaN,2026-01-31,2026-03-31,blinkit_d112_718287
4,NaN,0.0,NaN,2026-01-31,2026-04-30,blinkit_d112_718287
...,...,...,...,...,...,...
893075,NaN,0.0,NaN,2026-04-30,2026-08-31,zepto_nan_811287
893076,NaN,0.0,NaN,2026-04-30,2026-09-30,zepto_nan_811287
893077,NaN,0.0,NaN,2026-04-30,2026-10-31,zepto_nan_811287
893078,NaN,0.0,NaN,2026-04-30,2026-11-30,zepto_nan_811287


In [ ]:
tx.groupby(['Month Date'])[['Offtake Chain depot PSKU Forecast Vol',
        'Safety Stock Vol',
        'Actual Closing SOH Lag 1 Vol']].sum().reset_index()

,Month Date,Offtake Chain depot PSKU Forecast Vol,Safety Stock Vol,Actual Closing SOH Lag 1 Vol
0,2025-12-31,0.000000,32866.765522,0.000000
1,2026-01-31,102706.479134,67430.108364,96184.313361
2,2026-02-28,195622.034064,107051.669647,146695.398004
3,2026-03-31,349838.749092,148270.321541,233388.794134
4,2026-04-30,480555.607265,136641.815601,284609.244726
5,2026-05-31,464284.979695,119518.611277,265067.944611
6,2026-06-30,396101.130099,89021.713827,204835.248380
7,2026-07-31,314972.659027,53428.793464,0.000000
8,2026-08-31,199921.612923,17551.117108,0.000000
9,2026-09-30,73744.824946,7114.641679,0.000000


In [ ]:
# ### Move UP

# final_df['Calculated Primary Vol'] = np.where(
#     final_df['M month'] == 'M',
#     final_df['Secondary Plan Vol'],
#     final_df['Offtake Chain FC PSKU Forecast Vol'].fillna(0) + \
#         final_df['Safety Stock Vol'].fillna(0) - \
#         final_df['Assumed Closing SOH Lag 1 Vol'].fillna(0)
# )

In [ ]:
calculated_primary = []

for (_, _), group_df in tqdm(final_df.groupby(['Run Month', 'Key2'])):
    calculated_primary.extend(
        calculate_primary_iteratively(group_df)
    )

calculated_primary_df = pd.DataFrame(calculated_primary)

 10%|█████████▋                                                                                       | 8922/89308 [00:20<02:49, 475.11it/s]

100%|████████████████████████████████████████████████████████████████████████████████████████████████| 89308/89308 [03:16<00:00, 453.86it/s]


In [ ]:
del calculated_primary

In [ ]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    calculated_primary_df,
    on=['Run Month', 'Month Date', 'Key2'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [ ]:
final_df['Calculated Primary Vol'].min(), final_df['Final Assumed Closing SOH Vol'].min()

(0.0, 0.0)

In [ ]:
final_df.sort_values(by=['Run Month', 'Key2', 'Month Date'], inplace=True)

In [ ]:
final_df['Final Assumed Closing SOH Lag 1 Vol'] = final_df.groupby(
    ['Run Month', 'Key']
)['Final Assumed Closing SOH Vol'].shift(1)

In [ ]:
final_df.columns

Index(['Key2', 'Chain', 'Depot', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Primary Till Date Actuals Vol', 'Secondary Plan Vol',
       'Primary P3M Vol', 'Key', 'Offtake Chain Depot PSKU Vol', 'M month',
       'Offtake Chain depot PSKU Forecast Vol', 'Norms SOH', 'Norm Days',
       'Safety Stock Vol', 'Actual Closing SOH Vol',
       'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol',
       'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol',
       'Assumed Closing SOH Lag 2 Vol', 'Primary Actuals Vol',
       'Sec Actuals Vol', 'Primary P3M redundant Vol',
       'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M Vol',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
       'LY Primary Actuals Lead 2 Vol', 'Offtake Actuals Vol',
       'Offtake P3M Vol', 'LY Offtake Actuals Vol', 'LY Offtake P3M Vol',
       'LY Offtake Actuals Lag 1 Vol', 'LY Of

In [ ]:
final_df = final_df[columns_order]

In [ ]:
vol_to_val_cols = [col for col in final_df.columns if 'Vol' in col]
vol_to_val_cols

['Primary Till Date Actuals Vol',
 'Secondary Plan Vol',
 'Primary P3M Vol',
 'Offtake Chain Depot PSKU Vol',
 'Offtake Chain depot PSKU Forecast Vol',
 'Safety Stock Vol',
 'Actual Closing SOH Vol',
 'Actual Closing SOH Lag 1 Vol',
 'Actual Closing SOH Lag 2 Vol',
 'Assumed Closing SOH Vol',
 'Assumed Closing SOH Lag 1 Vol',
 'Assumed Closing SOH Lag 2 Vol',
 'Final Assumed Closing SOH Vol',
 'Final Assumed Closing SOH Lag 1 Vol',
 'Primary Actuals Vol',
 'Sec Actuals Vol',
 'Primary P3M redundant Vol',
 'LY Primary Actuals Vol',
 'LY Sec Actuals Vol',
 'LY Primary P3M Vol',
 'Offtake Actuals Vol',
 'Offtake P3M Vol',
 'LY Offtake Actuals Vol',
 'LY Offtake P3M Vol',
 'Primary Actuals Lag 1 Vol',
 'Primary Actuals Lag 2 Vol',
 'Primary Actuals Lag 3 Vol',
 'LY Primary Actuals Lag 1 Vol',
 'LY Primary Actuals Lag 2 Vol',
 'LY Primary Actuals Lag 3 Vol',
 'LY Primary Actuals Lead 1 Vol',
 'LY Primary Actuals Lead 2 Vol',
 'Offtake Actuals Lag 1 Vol',
 'Offtake Actuals Lag 2 Vol',
 'Offt

In [ ]:
for col in vol_to_val_cols:
    final_df[col[:-3] + 'Val'] = final_df[col].fillna(0) * final_df['Index Rate'] / (10 ** 7)

In [ ]:
final_df.duplicated(subset=['Run Month', 'Key2', 'Month Date']).sum()

0

In [ ]:
final_df

,Key2,Key,Chain,Depot,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,...,Offtake Chain depot PSKU Lag 3 Val,Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU Actuals Val,LY Offtake Chain depot PSKU Lag 1 Val,LY Offtake Chain depot PSKU Lag 2 Val,LY Offtake Chain depot PSKU Lag 3 Val,LY Offtake Chain depot PSKU Lead 1 Val,LY Offtake Chain depot PSKU Lead 2 Val,Calculated Primary Val
0,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-01-31,2025-12-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-01-31,2026-01-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-01-31,2026-02-28,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-01-31,2026-03-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-01-31,2026-04-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
893075,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-04-30,2026-08-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
893076,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-04-30,2026-09-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
893077,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-04-30,2026-10-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
893078,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-04-30,2026-11-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
final_df[(final_df['Month Date'] == '2026-10-31') & (final_df['Run Month'] == '2026-06-30')]['Calculated Primary Val'].sum()

0.0

In [ ]:
# final_df['forecast_error'] = final_df['Calculated Primary Val'] - final_df['Primary Actuals Val']
# final_df['abs_forecast_error'] = abs(final_df['forecast_error'])

In [ ]:
final_df.columns[:60]

Index(['Key2', 'Key', 'Chain', 'Depot', 'PSKU', 'Brand', 'Index Rate',
       'Portfolio', 'Run Month', 'Month Date', 'M month',
       'Primary Till Date Actuals Vol', 'Secondary Plan Vol',
       'Primary P3M Vol', 'Offtake Chain Depot PSKU Vol',
       'Offtake Chain depot PSKU Forecast Vol', 'Norms SOH', 'Norm Days',
       'Safety Stock Vol', 'Actual Closing SOH Vol',
       'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol',
       'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol',
       'Assumed Closing SOH Lag 2 Vol', 'Final Assumed Closing SOH Vol',
       'Final Assumed Closing SOH Lag 1 Vol', 'Primary Actuals Vol',
       'Sec Actuals Vol', 'Primary P3M redundant Vol',
       'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M Vol',
       'Offtake Actuals Vol', 'Offtake P3M Vol', 'LY Offtake Actuals Vol',
       'LY Offtake P3M Vol', 'Primary Actuals Lag 1 Vol',
       'Primary Actuals Lag 2 Vol', 'Primary Actuals Lag 3 Vol',
       'LY Pri

In [ ]:
final_df[final_df['Month Date'] == '2026-05-31']['Primary Actuals Vol'].sum()

593444.1

In [ ]:
# final_df.to_csv('qcom_chain_depot_psku_primary_forecast.csv')

### Depot PSKU

In [ ]:
offtake_columns = [
    col for col in final_df.columns 
    if ('offtake' in col.lower()) &
    ('vol' in col.lower()) & ('depot' in col.lower())
]

for col in ['Offtake Chain Depot PSKU Vol',  'Offtake Actuals Vol']:
    if col in offtake_columns:
        offtake_columns.remove(col)
    
offtake_columns

['Offtake Chain depot PSKU Forecast Vol',
 'Offtake Chain depot PSKU Lag 1 Vol',
 'Offtake Chain depot PSKU Lag 2 Vol',
 'Offtake Chain depot PSKU Lag 3 Vol',
 'Offtake Chain depot PSKU P3M Vol',
 'LY Offtake Chain depot PSKU P3M Vol',
 'LY Offtake Chain depot PSKU Actuals Vol',
 'LY Offtake Chain depot PSKU Lag 1 Vol',
 'LY Offtake Chain depot PSKU Lag 2 Vol',
 'LY Offtake Chain depot PSKU Lag 3 Vol',
 'LY Offtake Chain depot PSKU Lead 1 Vol',
 'LY Offtake Chain depot PSKU Lead 2 Vol']

In [ ]:
soh_cols = [
    col for col in final_df.columns 
    if ('soh' in col.lower()) &
    ('closing' in col.lower()) &
    ('vol' in col.lower())
]
soh_cols

['Actual Closing SOH Vol',
 'Actual Closing SOH Lag 1 Vol',
 'Actual Closing SOH Lag 2 Vol',
 'Assumed Closing SOH Vol',
 'Assumed Closing SOH Lag 1 Vol',
 'Assumed Closing SOH Lag 2 Vol',
 'Final Assumed Closing SOH Vol',
 'Final Assumed Closing SOH Lag 1 Vol']

In [ ]:
final_df.columns

Index(['Key2', 'Key', 'Chain', 'Depot', 'PSKU', 'Brand', 'Index Rate',
       'Portfolio', 'Run Month', 'Month Date',
       ...
       'Offtake Chain depot PSKU Lag 3 Val',
       'Offtake Chain depot PSKU P3M Val',
       'LY Offtake Chain depot PSKU P3M Val',
       'LY Offtake Chain depot PSKU Actuals Val',
       'LY Offtake Chain depot PSKU Lag 1 Val',
       'LY Offtake Chain depot PSKU Lag 2 Val',
       'LY Offtake Chain depot PSKU Lag 3 Val',
       'LY Offtake Chain depot PSKU Lead 1 Val',
       'LY Offtake Chain depot PSKU Lead 2 Val', 'Calculated Primary Val'],
      dtype='object', length=117)

In [ ]:
base_df = final_df.copy()
base_df = base_df[
    base_df['Depot'].notna()
]

In [ ]:
base_df.shape

(852000, 117)

In [ ]:
base_df['Depot'].isna().sum()

0

In [ ]:
base_df = base_df.groupby(
    ['Run Month', 'Month Date', 'M month', 'Depot', 'PSKU', 
     'Brand', 'Portfolio', 'Index Rate'], as_index=False
)[['Calculated Primary Vol'] + offtake_columns + soh_cols + ['Safety Stock Vol']].sum()

In [ ]:
base_df['Offtake Chain depot PSKU Forecast Vol'].sum()

2662402.175308273

In [ ]:
depot_psku_primary_query = """
SELECT
    CM.depot_code,
    MM.parent_material_code,
    MM.material_group_code,
    LAST_DAY(MESR.month_date) AS month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain_type = 'E Com B2C' AND 
        chain IN ('Grofers', 'Zepto', 'Kiranakart Technologies', 'Swiggy', 'ZEPTO')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
JOIN
(
    SELECT DISTINCT
        customer_code,
        depot_code
    FROM
        mst_customer
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) CM on MESR.distributor_code = CM.customer_code
WHERE month_date > '2022-12-31'
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 2, 4
"""

depot_psku_primary_df = pd.read_sql(
    depot_psku_primary_query,
    prod_conn
)

In [ ]:
depot_psku_primary_df.head()

,DEPOT_CODE,PARENT_MATERIAL_CODE,MATERIAL_GROUP_CODE,MONTH_DATE,PRI_ACTUALS_VOL_RUM,PRI_APO_PLAN_VOL_RUM,SEC_APO_PLAN_VOL_RUM,SEC_ACTUALS_VOL_RUM
0,D112,718472,ADV-AHO-R,2023-02-28,0.0,0.0,0.044,0.0
1,D112,718472,ADV-AHO-R,2023-03-31,0.0,0.0,0.027,0.0
2,D112,718472,ADV-AHO-R,2023-04-30,0.0,0.0,0.044,0.0
3,D112,718472,ADV-AHO-R,2023-06-30,0.0,0.0,0.147,0.0
4,D112,718472,ADV-AHO-R,2023-07-31,0.0,0.0,0.056,0.0


In [ ]:
depot_psku_primary_df.columns = depot_psku_primary_df.columns.str.lower()
depot_psku_primary_df['parent_material_code'] = depot_psku_primary_df['parent_material_code'].astype(int)

In [ ]:
depot_psku_primary_df['sec_actuals_vol_rum'].sum()

2548089.0270000007

In [ ]:
depot_psku_primary_df.groupby(['month_date'])['sec_actuals_vol_rum'].sum().reset_index()

,month_date,sec_actuals_vol_rum
0,2023-01-31,14308.209
1,2023-02-28,22359.437
2,2023-03-31,24087.087
3,2023-04-30,12969.899
4,2023-05-31,18067.597
5,2023-06-30,17244.321
6,2023-07-31,22721.000
7,2023-08-31,24967.529
8,2023-09-30,18266.147
9,2023-10-31,26286.404


In [ ]:
depot_psku_primary_df[depot_psku_primary_df['month_date'] == '2026-05-31']['sec_actuals_vol_rum'].sum()

0.0

In [ ]:
depot_psku_primary_df = realign_pskus(depot_psku_primary_df.copy(), 'parent_material_code')

In [ ]:
depot_psku_primary_df = depot_psku_primary_df.groupby(
    ['depot_code', 'parent_material_code', 'material_group_code', 'month_date'], as_index=False, dropna=False
).sum()

In [ ]:
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['parent_material_code'] != 715096]

In [ ]:
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['material_group_code']!='PABABY_SP']
depot_psku_primary_df = depot_psku_primary_df[~depot_psku_primary_df['material_group_code'].isin(['LVNPST_ML', 'PABABY_GM', 'PADV_WIPS'])]

In [ ]:
depot_psku_primary_df.duplicated(subset=['depot_code', 'parent_material_code', 'month_date']).sum()

0

In [ ]:
depot_psku_primary_df = impute_missing_dates(
    depot_psku_primary_df.copy(),
    key=['depot_code', 'parent_material_code'],
    date_col='month_date'
)

11244it [00:02, 4083.72it/s]


In [ ]:
cols = ['depot_code', 'parent_material_code', 'material_group_code']

depot_psku_primary_df[cols] = depot_psku_primary_df.groupby('key')[cols].transform(lambda x: x.ffill().bfill())

In [ ]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    depot_psku_primary_df[col] = depot_psku_primary_df[col].fillna(0)

In [ ]:
depot_psku_primary_df['parent_material_code'] = depot_psku_primary_df['parent_material_code'].astype(int)

In [ ]:
depot_psku_primary_df.duplicated(subset=['key', 'month_date']).sum()

0

In [ ]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if depot_psku_primary_df[col].min() < 0:
        print(col)

for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    depot_psku_primary_df[col] = depot_psku_primary_df[col].clip(lower=0)

pri_actuals_vol_rum
sec_actuals_vol_rum


In [ ]:
depot_psku_primary_df.sort_values(by=['key', 'month_date'], inplace=True)

In [ ]:
depot_psku_primary_df['Primary P3M Vol'] = depot_psku_primary_df.groupby(
    ['key'], 
    as_index = False, group_keys = False
)['pri_actuals_vol_rum'].shift(1).rolling(window=3, min_periods=1).mean()

In [ ]:
depot_psku_primary_df['Primary Actuals Lag 1 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(1)

depot_psku_primary_df['Primary Actuals Lag 2 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(2)

depot_psku_primary_df['Primary Actuals Lag 3 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(3)

depot_psku_primary_df['LY Primary Actuals Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(12)

depot_psku_primary_df['LY Primary Actuals Lag 1 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(13)

depot_psku_primary_df['LY Primary Actuals Lag 2 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(14)

depot_psku_primary_df['LY Primary Actuals Lag 3 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(15)

depot_psku_primary_df['LY Primary Actuals Lead 1 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(11)

depot_psku_primary_df['LY Primary Actuals Lead 2 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(10)

In [ ]:
depot_psku_primary_df['LY Primary P3M Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['Primary P3M Vol'].shift(12)

In [ ]:
depot_psku_primary_df = depot_psku_primary_df.rename(columns={
    'key': 'Key',
    'depot_code': 'Depot',
    'parent_material_code': 'PSKU',
    'material_group_code': 'Brand',
    'month_date': 'Month Date',
    'pri_actuals_vol_rum': 'Primary Actuals Vol',
    'pri_apo_plan_vol_rum': 'Primary Plan Vol',
    'sec_actuals_vol_rum': 'Secondary Actuals Vol',
    'sec_apo_plan_vol_rum': 'Secondary Plan Vol'
})

In [ ]:
final_depot_psku_df = depot_psku_primary_df[['Key', 'Depot', 'PSKU', 'Brand']].drop_duplicates()

In [ ]:
final_df['Run Month'].unique()

<DatetimeArray>
['2026-01-31 00:00:00', '2026-02-28 00:00:00', '2026-03-31 00:00:00',
 '2026-04-30 00:00:00']
Length: 4, dtype: datetime64[ns]

In [ ]:
tmp_df = pd.DataFrame()

for rm in run_months_list: 
    mth_dates = [pd.to_datetime(rm) + MonthEnd(i) for i in range(-1, 9)]
    for mth_dt in mth_dates:
        tmp_df2 = final_depot_psku_df.copy()
        tmp_df2['Run Month'] = pd.to_datetime(rm)
        tmp_df2['Month Date'] = pd.to_datetime(mth_dt)

        tmp_df = pd.concat([tmp_df, tmp_df2], ignore_index=True)
        del tmp_df2

final_depot_psku_df = tmp_df.copy()    
del tmp_df

In [ ]:
final_depot_psku_df['Month Date'].max()

Timestamp('2026-12-31 00:00:00')

In [ ]:
base_df['Calculated Primary Vol'].sum()

2463033.73857853

In [ ]:
base_df[['Depot', 'PSKU', 'Run Month', 'Month Date']].dtypes

Depot                 object
PSKU                   int64
Run Month     datetime64[ns]
Month Date    datetime64[ns]
dtype: object

In [ ]:
final_depot_psku_df['Depot'] = final_depot_psku_df['Depot'].str.lower()

In [ ]:
base_df['PSKU'] = base_df['PSKU'].astype(int)
final_depot_psku_df['PSKU'] = final_depot_psku_df['PSKU'].astype(int)


In [ ]:
len_before_merge = len(final_depot_psku_df)
final_depot_psku_df = final_depot_psku_df.merge(
    base_df.drop(['M month', 'Brand', 'Portfolio', 'Index Rate'], axis=1), 
    on=['Depot', 'PSKU', 'Run Month', 'Month Date'],
    how='left'
)
assert len_before_merge == len(final_depot_psku_df)
del len_before_merge

In [ ]:
final_depot_psku_df['Calculated Primary Vol'].sum()

2463033.7385785305

In [ ]:
final_depot_psku_df['Calculated Primary Vol'] = final_depot_psku_df['Calculated Primary Vol'].fillna(0)

In [ ]:
depot_psku_primary_df.columns

Index(['Month Date', 'Key', 'Depot', 'PSKU', 'Brand', 'Primary Actuals Vol',
       'Primary Plan Vol', 'Secondary Plan Vol', 'Secondary Actuals Vol',
       'Primary P3M Vol', 'Primary Actuals Lag 1 Vol',
       'Primary Actuals Lag 2 Vol', 'Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Vol', 'LY Primary Actuals Lag 1 Vol',
       'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Lead 1 Vol', 'LY Primary Actuals Lead 2 Vol',
       'LY Primary P3M Vol'],
      dtype='object')

In [ ]:
depot_psku_primary_df['Depot'] = depot_psku_primary_df['Depot'].str.lower()

In [ ]:
len_before_merge = len(final_depot_psku_df)
final_depot_psku_df = final_depot_psku_df.merge(
    depot_psku_primary_df[['Depot', 'PSKU', 'Month Date', 'Primary Actuals Vol',
       'Primary Plan Vol', 'Secondary Plan Vol', 'Secondary Actuals Vol',
       'Primary P3M Vol', 'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol',
       'Primary Actuals Lag 3 Vol', 'LY Primary Actuals Vol', 'LY Primary Actuals Lag 1 Vol',
       'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol', 
       'LY Primary Actuals Lead 2 Vol',
       'LY Primary P3M Vol']], 
    on=['Depot', 'PSKU', 'Month Date'],
    how='left'
)
assert len_before_merge == len(final_depot_psku_df)
del len_before_merge

In [ ]:
final_depot_psku_df.columns

Index(['Key', 'Depot', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Calculated Primary Vol', 'Offtake Chain depot PSKU Forecast Vol',
       'Offtake Chain depot PSKU Lag 1 Vol',
       'Offtake Chain depot PSKU Lag 2 Vol',
       'Offtake Chain depot PSKU Lag 3 Vol',
       'Offtake Chain depot PSKU P3M Vol',
       'LY Offtake Chain depot PSKU P3M Vol',
       'LY Offtake Chain depot PSKU Actuals Vol',
       'LY Offtake Chain depot PSKU Lag 1 Vol',
       'LY Offtake Chain depot PSKU Lag 2 Vol',
       'LY Offtake Chain depot PSKU Lag 3 Vol',
       'LY Offtake Chain depot PSKU Lead 1 Vol',
       'LY Offtake Chain depot PSKU Lead 2 Vol', 'Actual Closing SOH Vol',
       'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol',
       'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol',
       'Assumed Closing SOH Lag 2 Vol', 'Final Assumed Closing SOH Vol',
       'Final Assumed Closing SOH Lag 1 Vol', 'Safety Stock Vol',
       'Primary Actuals Vol', 'Primary P

In [ ]:
final_depot_psku_df.sort_values(by=['Run Month', 'Key', 'Month Date'], inplace=True)

In [ ]:
for col in ['Primary P3M Vol', 'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol', 
            'Primary Actuals Lag 3 Vol', 'LY Primary P3M Vol']:
    # if not 'LY' in col:  'LY P6M',
    final_depot_psku_df.loc[final_depot_psku_df['Month Date'] > final_depot_psku_df['Run Month'], [col]] = np.nan
    final_depot_psku_df[col] = final_depot_psku_df.groupby(['Run Month', 'Key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [ ]:
vol_to_val_cols = [col for col in final_depot_psku_df.columns if ('Vol' in col)]
vol_to_val_cols

['Calculated Primary Vol',
 'Offtake Chain depot PSKU Forecast Vol',
 'Offtake Chain depot PSKU Lag 1 Vol',
 'Offtake Chain depot PSKU Lag 2 Vol',
 'Offtake Chain depot PSKU Lag 3 Vol',
 'Offtake Chain depot PSKU P3M Vol',
 'LY Offtake Chain depot PSKU P3M Vol',
 'LY Offtake Chain depot PSKU Actuals Vol',
 'LY Offtake Chain depot PSKU Lag 1 Vol',
 'LY Offtake Chain depot PSKU Lag 2 Vol',
 'LY Offtake Chain depot PSKU Lag 3 Vol',
 'LY Offtake Chain depot PSKU Lead 1 Vol',
 'LY Offtake Chain depot PSKU Lead 2 Vol',
 'Actual Closing SOH Vol',
 'Actual Closing SOH Lag 1 Vol',
 'Actual Closing SOH Lag 2 Vol',
 'Assumed Closing SOH Vol',
 'Assumed Closing SOH Lag 1 Vol',
 'Assumed Closing SOH Lag 2 Vol',
 'Final Assumed Closing SOH Vol',
 'Final Assumed Closing SOH Lag 1 Vol',
 'Safety Stock Vol',
 'Primary Actuals Vol',
 'Primary Plan Vol',
 'Secondary Plan Vol',
 'Secondary Actuals Vol',
 'Primary P3M Vol',
 'Primary Actuals Lag 1 Vol',
 'Primary Actuals Lag 2 Vol',
 'Primary Actuals Lag 3

In [ ]:
len_before_merge = len(final_depot_psku_df)
final_depot_psku_df = final_depot_psku_df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(columns={
        'brand_code': 'Brand',
        'qtr_ind_rate': 'Index Rate'
    }),
    on=['Brand'],
    how='left'
)
assert len_before_merge == len(final_depot_psku_df)
del len_before_merge

In [ ]:
for col in vol_to_val_cols:
    final_depot_psku_df[col[:-3] + 'Val'] = final_depot_psku_df[col] * final_depot_psku_df['Index Rate'] / (10 ** 7)

In [ ]:
final_depot_psku_df['M Month'] = final_depot_psku_df.apply(
    lambda x: mappings[x['Run Month']].get(x['Month Date'], np.nan),
    axis=1
)

In [ ]:
final_depot_psku_df = final_depot_psku_df[final_depot_psku_df['M Month'].notna()]

In [ ]:
final_depot_psku_df.columns

Index(['Key', 'Depot', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Calculated Primary Vol', 'Offtake Chain depot PSKU Forecast Vol',
       'Offtake Chain depot PSKU Lag 1 Vol',
       'Offtake Chain depot PSKU Lag 2 Vol',
       'Offtake Chain depot PSKU Lag 3 Vol',
       'Offtake Chain depot PSKU P3M Vol',
       'LY Offtake Chain depot PSKU P3M Vol',
       'LY Offtake Chain depot PSKU Actuals Vol',
       'LY Offtake Chain depot PSKU Lag 1 Vol',
       'LY Offtake Chain depot PSKU Lag 2 Vol',
       'LY Offtake Chain depot PSKU Lag 3 Vol',
       'LY Offtake Chain depot PSKU Lead 1 Vol',
       'LY Offtake Chain depot PSKU Lead 2 Vol', 'Actual Closing SOH Vol',
       'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol',
       'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol',
       'Assumed Closing SOH Lag 2 Vol', 'Final Assumed Closing SOH Vol',
       'Final Assumed Closing SOH Lag 1 Vol', 'Safety Stock Vol',
       'Primary Actuals Vol', 'Primary P

In [ ]:
len_before_merge = len(final_depot_psku_df)
final_depot_psku_df = final_depot_psku_df.merge(
    brand_md_df.rename(
        columns={'brand_code': 'Brand'}
    ),
    on=['Brand'], 
    how='left'
)
assert len_before_merge == len(final_depot_psku_df)
del len_before_merge

In [ ]:
final_depot_psku_df.rename(columns={'portfolio': 'Portfolio'}, inplace=True)

In [ ]:
final_depot_psku_df.columns

Index(['Key', 'Depot', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Calculated Primary Vol', 'Offtake Chain depot PSKU Forecast Vol',
       'Offtake Chain depot PSKU Lag 1 Vol',
       'Offtake Chain depot PSKU Lag 2 Vol',
       'Offtake Chain depot PSKU Lag 3 Vol',
       'Offtake Chain depot PSKU P3M Vol',
       'LY Offtake Chain depot PSKU P3M Vol',
       'LY Offtake Chain depot PSKU Actuals Vol',
       'LY Offtake Chain depot PSKU Lag 1 Vol',
       'LY Offtake Chain depot PSKU Lag 2 Vol',
       'LY Offtake Chain depot PSKU Lag 3 Vol',
       'LY Offtake Chain depot PSKU Lead 1 Vol',
       'LY Offtake Chain depot PSKU Lead 2 Vol', 'Actual Closing SOH Vol',
       'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol',
       'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol',
       'Assumed Closing SOH Lag 2 Vol', 'Final Assumed Closing SOH Vol',
       'Final Assumed Closing SOH Lag 1 Vol', 'Safety Stock Vol',
       'Primary Actuals Vol', 'Primary P

In [ ]:
columns_order = [
    'Key', 'Depot', 'PSKU', 'Brand', 'Portfolio', 'Index Rate', 
    'Run Month', 'Month Date', 'M Month',
    'Primary Actuals Vol',

    'Calculated Primary Vol', 
    
    'Secondary Plan Vol', 'Primary Actuals Lag 1 Vol',
    'Primary Actuals Lag 2 Vol', 'Primary Actuals Lag 3 Vol',
    'LY Primary Actuals Vol', 'Primary P3M Vol', 'LY Primary P3M Vol',
    'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
    'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol', 'LY Primary Actuals Lead 2 Vol',
    
    
    'Offtake Chain depot PSKU Forecast Vol',
    'Offtake Chain depot PSKU Lag 1 Vol', 'Offtake Chain depot PSKU Lag 2 Vol',
    'Offtake Chain depot PSKU Lag 3 Vol', 'LY Offtake Chain depot PSKU Actuals Vol',
    'Offtake Chain depot PSKU P3M Vol', 'LY Offtake Chain depot PSKU P3M Vol',
    'LY Offtake Chain depot PSKU Lag 1 Vol', 'LY Offtake Chain depot PSKU Lag 2 Vol',
    'LY Offtake Chain depot PSKU Lag 3 Vol', 'LY Offtake Chain depot PSKU Lead 1 Vol',
    'LY Offtake Chain depot PSKU Lead 2 Vol', 

    'Actual Closing SOH Vol', 'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol', 
    'Final Assumed Closing SOH Vol', 'Final Assumed Closing SOH Lag 1 Vol',
    'Safety Stock Vol',
    'Primary Actuals Val',
    'Calculated Primary Val', 

    'Secondary Plan Val', 'Primary Actuals Lag 1 Val',
    'Primary Actuals Lag 2 Val', 'Primary Actuals Lag 3 Val',
    'LY Primary Actuals Val', 'Primary P3M Val', 'LY Primary P3M Val',
    'LY Primary Actuals Lag 1 Val', 'LY Primary Actuals Lag 2 Val',
    'LY Primary Actuals Lag 3 Val', 'LY Primary Actuals Lead 1 Val', 'LY Primary Actuals Lead 2 Val',

    
    'Offtake Chain depot PSKU Forecast Val',
    'Offtake Chain depot PSKU Lag 1 Val', 'Offtake Chain depot PSKU Lag 2 Val',
    'Offtake Chain depot PSKU Lag 3 Val', 'LY Offtake Chain depot PSKU Actuals Val', 
    'Offtake Chain depot PSKU P3M Val', 'LY Offtake Chain depot PSKU P3M Val', 
    'LY Offtake Chain depot PSKU Lag 1 Val', 'LY Offtake Chain depot PSKU Lag 2 Val',
    'LY Offtake Chain depot PSKU Lag 3 Val', 'LY Offtake Chain depot PSKU Lead 1 Val',
    'LY Offtake Chain depot PSKU Lead 2 Val', 
    
    'Actual Closing SOH Val', 'Actual Closing SOH Lag 1 Val', 'Actual Closing SOH Lag 2 Val', 
    'Final Assumed Closing SOH Val', 'Final Assumed Closing SOH Lag 1 Val', 'Safety Stock Val', 
]
    # 'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol', 'Assumed Closing SOH Lag 2 Vol', 
    # 'Assumed Closing SOH Val', 'Assumed Closing SOH Lag 1 Val', 'Assumed Closing SOH Lag 2 Val', 

In [ ]:
for col in final_depot_psku_df.columns:
    if col not in columns_order:
        print(col)

Assumed Closing SOH Vol
Assumed Closing SOH Lag 1 Vol
Assumed Closing SOH Lag 2 Vol
Primary Plan Vol
Secondary Actuals Vol
Assumed Closing SOH Val
Assumed Closing SOH Lag 1 Val
Assumed Closing SOH Lag 2 Val
Primary Plan Val
Secondary Actuals Val


In [ ]:
final_depot_psku_df = final_depot_psku_df[columns_order]

In [ ]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Portfolio,Index Rate,Run Month,Month Date,M Month,Primary Actuals Vol,...,LY Offtake Chain depot PSKU Lag 2 Val,LY Offtake Chain depot PSKU Lag 3 Val,LY Offtake Chain depot PSKU Lead 1 Val,LY Offtake Chain depot PSKU Lead 2 Val,Actual Closing SOH Val,Actual Closing SOH Lag 1 Val,Actual Closing SOH Lag 2 Val,Final Assumed Closing SOH Val,Final Assumed Closing SOH Lag 1 Val,Safety Stock Val
0,D112_715098,d112,715098,CO_SO_PCP,Skin Care,1220.081,2026-01-31,2026-01-31,M,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,D112_715098,d112,715098,CO_SO_PCP,Skin Care,1220.081,2026-01-31,2026-02-28,M+1,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,D112_715098,d112,715098,CO_SO_PCP,Skin Care,1220.081,2026-01-31,2026-03-31,M+2,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,D112_715098,d112,715098,CO_SO_PCP,Skin Care,1220.081,2026-01-31,2026-04-30,M+3,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,D112_715098,d112,715098,CO_SO_PCP,Skin Care,1220.081,2026-01-31,2026-05-31,M+4,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
404779,D677_811287,d677,811287,PA_RSW_SR,NaN,740.000,2026-04-30,2026-08-31,M+4,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
404780,D677_811287,d677,811287,PA_RSW_SR,NaN,740.000,2026-04-30,2026-09-30,M+5,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
404781,D677_811287,d677,811287,PA_RSW_SR,NaN,740.000,2026-04-30,2026-10-31,M+6,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
404782,D677_811287,d677,811287,PA_RSW_SR,NaN,740.000,2026-04-30,2026-11-30,M+7,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
final_depot_psku_df.isna().sum()

Key                                        0
Depot                                      0
PSKU                                       0
Brand                                      0
Portfolio                              23256
                                       ...  
Actual Closing SOH Lag 1 Val           79272
Actual Closing SOH Lag 2 Val           79272
Final Assumed Closing SOH Val          79272
Final Assumed Closing SOH Lag 1 Val    79272
Safety Stock Val                       79272
Length: 73, dtype: int64

In [ ]:
final_df['Calculated Primary Val'].sum(),final_depot_psku_df['Calculated Primary Val'].sum()

(663.6864986339642, 663.6864986339646)

In [782]:
final_depot_psku_df.columns[30:]

Index(['LY Offtake Chain depot PSKU Lag 1 Vol',
       'LY Offtake Chain depot PSKU Lag 2 Vol',
       'LY Offtake Chain depot PSKU Lag 3 Vol',
       'LY Offtake Chain depot PSKU Lead 1 Vol',
       'LY Offtake Chain depot PSKU Lead 2 Vol', 'Actual Closing SOH Vol',
       'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol',
       'Final Assumed Closing SOH Vol', 'Final Assumed Closing SOH Lag 1 Vol',
       'Safety Stock Vol', 'Primary Actuals Val', 'Calculated Primary Val',
       'Secondary Plan Val', 'Primary Actuals Lag 1 Val',
       'Primary Actuals Lag 2 Val', 'Primary Actuals Lag 3 Val',
       'LY Primary Actuals Val', 'Primary P3M Val', 'LY Primary P3M Val',
       'LY Primary Actuals Lag 1 Val', 'LY Primary Actuals Lag 2 Val',
       'LY Primary Actuals Lag 3 Val', 'LY Primary Actuals Lead 1 Val',
       'LY Primary Actuals Lead 2 Val',
       'Offtake Chain depot PSKU Forecast Val',
       'Offtake Chain depot PSKU Lag 1 Val',
       'Offtake Chain depot PSKU 

In [789]:
final_depot_psku_df[(final_depot_psku_df['Run Month'] == '2026-02-28') & (final_depot_psku_df['Month Date'] == '2026-03-31')]['Calculated Primary Val'].sum()

32.94662405038807

### SAVE

In [786]:
today_dt = datetime.today()
today_dt.strftime('%d-%b-%Y')

'01-Jul-2026'

In [787]:
VERSION = 'Backtest'

In [790]:
os.makedirs(f'QCOM Chain PSKU OTP Output/{VERSION}')

In [791]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Portfolio,Index Rate,Run Month,Month Date,M Month,Primary Actuals Vol,...,LY Offtake Chain depot PSKU Lag 2 Val,LY Offtake Chain depot PSKU Lag 3 Val,LY Offtake Chain depot PSKU Lead 1 Val,LY Offtake Chain depot PSKU Lead 2 Val,Actual Closing SOH Val,Actual Closing SOH Lag 1 Val,Actual Closing SOH Lag 2 Val,Final Assumed Closing SOH Val,Final Assumed Closing SOH Lag 1 Val,Safety Stock Val
0,D112_715098,d112,715098,CO_SO_PCP,Skin Care,1220.081,2026-01-31,2026-01-31,M,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,D112_715098,d112,715098,CO_SO_PCP,Skin Care,1220.081,2026-01-31,2026-02-28,M+1,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,D112_715098,d112,715098,CO_SO_PCP,Skin Care,1220.081,2026-01-31,2026-03-31,M+2,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,D112_715098,d112,715098,CO_SO_PCP,Skin Care,1220.081,2026-01-31,2026-04-30,M+3,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,D112_715098,d112,715098,CO_SO_PCP,Skin Care,1220.081,2026-01-31,2026-05-31,M+4,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
404779,D677_811287,d677,811287,PA_RSW_SR,NaN,740.000,2026-04-30,2026-08-31,M+4,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
404780,D677_811287,d677,811287,PA_RSW_SR,NaN,740.000,2026-04-30,2026-09-30,M+5,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
404781,D677_811287,d677,811287,PA_RSW_SR,NaN,740.000,2026-04-30,2026-10-31,M+6,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
404782,D677_811287,d677,811287,PA_RSW_SR,NaN,740.000,2026-04-30,2026-11-30,M+7,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [792]:
final_depot_psku_df.columns

Index(['Key', 'Depot', 'PSKU', 'Brand', 'Portfolio', 'Index Rate', 'Run Month',
       'Month Date', 'M Month', 'Primary Actuals Vol',
       'Calculated Primary Vol', 'Secondary Plan Vol',
       'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol',
       'Primary Actuals Lag 3 Vol', 'LY Primary Actuals Vol',
       'Primary P3M Vol', 'LY Primary P3M Vol', 'LY Primary Actuals Lag 1 Vol',
       'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Lead 1 Vol', 'LY Primary Actuals Lead 2 Vol',
       'Offtake Chain depot PSKU Forecast Vol',
       'Offtake Chain depot PSKU Lag 1 Vol',
       'Offtake Chain depot PSKU Lag 2 Vol',
       'Offtake Chain depot PSKU Lag 3 Vol',
       'LY Offtake Chain depot PSKU Actuals Vol',
       'Offtake Chain depot PSKU P3M Vol',
       'LY Offtake Chain depot PSKU P3M Vol',
       'LY Offtake Chain depot PSKU Lag 1 Vol',
       'LY Offtake Chain depot PSKU Lag 2 Vol',
       'LY Offtake Chain depot PSKU Lag 3

In [794]:
final_depot_psku_df['forecast_error'] = final_depot_psku_df['Calculated Primary Val'] - final_depot_psku_df['Primary Actuals Val']
final_depot_psku_df['abs_forecast_error'] = abs(final_depot_psku_df['forecast_error'])

In [795]:
final_depot_psku_df.to_csv(f'QCOM Chain PSKU OTP Output/{VERSION}/QCOM Depot PSKU Primary_{VERSION}_{today_dt.strftime("%d_%b_%Y")}.csv', index=False)
final_df.to_csv(f'QCOM Chain PSKU OTP Output/{VERSION}/QCOM Chain Depot PSKU Primary_{VERSION}_{today_dt.strftime("%d_%b_%Y")}.csv', index=False)

### chain forecast

In [645]:
chain_forecast = pd.read_csv('/data/aman_singh/acuuracy_check/Marico Ltd._forecast_Jun 2026_to_Sep 2026.csv')
chain_forecast

,facility_id,facility_name,item_id,item_name,manufacturer_id,manufacturer_name,vendor_id,vendor_name,Jun - forecast,Jul - forecast,Aug - forecast,Sep - forecast
0,3164,Mumbai M11 - Feeder Warehouse,10221228,Just Herbs 12 Chemical Free Nail Paint (Hot Re...,1385,Marico Ltd.,5271.0,Marico Ltd,24,25,38,37
1,3164,Mumbai M11 - Feeder Warehouse,10000063,Saffola Masala & Coriander Oats(Pack)38 gm - R...,1385,Marico Ltd.,5271.0,Marico Ltd,1,1,1,1
2,3164,Mumbai M11 - Feeder Warehouse,10000367,Saffola Masala Veggie Twist Oats(Pouch)38 gm -...,1385,Marico Ltd.,5271.0,Marico Ltd,4001,5292,5392,5409
3,3164,Mumbai M11 - Feeder Warehouse,10162271,Saffola Gold Unflavored Multigrain Oats (with ...,1385,Marico Ltd.,5271.0,Marico Ltd,15,15,15,15
4,3164,Mumbai M11 - Feeder Warehouse,10010052,Revive Fabric Stiffener(Pack)400 gm - Rs 160.0,1385,Marico Ltd.,5271.0,Marico Ltd,665,690,785,862
...,...,...,...,...,...,...,...,...,...,...,...,...
5836,5096,Faridabad - Feeder Warehouse,10221233,Just Herbs 12 Chemical Free Nail Paint (Gleami...,1385,Marico Ltd.,5271.0,Marico Ltd,10,10,10,10
5837,5096,Faridabad - Feeder Warehouse,10241144,Just Herbs Ayurvedic Mini Lipstick Kit(Box)9.6...,1385,Marico Ltd.,5271.0,Marico Ltd,66,64,70,68
5838,5096,Faridabad - Feeder Warehouse,10000910,Saffola Sunflower and Rice Bran Blended Cookin...,1385,Marico Ltd.,5271.0,Marico Ltd,1,1,1,1
5839,5096,Faridabad - Feeder Warehouse,10147561,Just Herbs Party Ready Nail Paint Kit(Packet)1...,1385,Marico Ltd.,5271.0,Marico Ltd,11,12,10,11


In [646]:

# Define the date columns and their corresponding dates
date_mapping = {
    'Jun - forecast': '2026-06-30',
    'Jul - forecast': '2026-07-31',
    'Aug - forecast': '2026-08-31',
    'Sep - forecast': '2026-09-30'
}

# Get all non-forecast columns
id_cols = [col for col in chain_forecast.columns if col not in date_mapping.keys()]

# Unpivot the dataframe
chain_forecast_unpivoted = chain_forecast.melt(
    id_vars=id_cols,
    value_vars=list(date_mapping.keys()),
    var_name='forecast_month',
    value_name='forecast_quantity'
)

# Map the forecast month to actual dates
chain_forecast_unpivoted['date'] = chain_forecast_unpivoted['forecast_month'].map(date_mapping)
chain_forecast_unpivoted['date'] = pd.to_datetime(chain_forecast_unpivoted['date'])

# Drop the temporary forecast_month column
chain_forecast_unpivoted = chain_forecast_unpivoted.drop('forecast_month', axis=1)

chain_forecast_unpivoted

,facility_id,facility_name,item_id,item_name,manufacturer_id,manufacturer_name,vendor_id,vendor_name,forecast_quantity,date
0,3164,Mumbai M11 - Feeder Warehouse,10221228,Just Herbs 12 Chemical Free Nail Paint (Hot Re...,1385,Marico Ltd.,5271.0,Marico Ltd,24,2026-06-30
1,3164,Mumbai M11 - Feeder Warehouse,10000063,Saffola Masala & Coriander Oats(Pack)38 gm - R...,1385,Marico Ltd.,5271.0,Marico Ltd,1,2026-06-30
2,3164,Mumbai M11 - Feeder Warehouse,10000367,Saffola Masala Veggie Twist Oats(Pouch)38 gm -...,1385,Marico Ltd.,5271.0,Marico Ltd,4001,2026-06-30
3,3164,Mumbai M11 - Feeder Warehouse,10162271,Saffola Gold Unflavored Multigrain Oats (with ...,1385,Marico Ltd.,5271.0,Marico Ltd,15,2026-06-30
4,3164,Mumbai M11 - Feeder Warehouse,10010052,Revive Fabric Stiffener(Pack)400 gm - Rs 160.0,1385,Marico Ltd.,5271.0,Marico Ltd,665,2026-06-30
...,...,...,...,...,...,...,...,...,...,...
23359,5096,Faridabad - Feeder Warehouse,10221233,Just Herbs 12 Chemical Free Nail Paint (Gleami...,1385,Marico Ltd.,5271.0,Marico Ltd,10,2026-09-30
23360,5096,Faridabad - Feeder Warehouse,10241144,Just Herbs Ayurvedic Mini Lipstick Kit(Box)9.6...,1385,Marico Ltd.,5271.0,Marico Ltd,68,2026-09-30
23361,5096,Faridabad - Feeder Warehouse,10000910,Saffola Sunflower and Rice Bran Blended Cookin...,1385,Marico Ltd.,5271.0,Marico Ltd,1,2026-09-30
23362,5096,Faridabad - Feeder Warehouse,10147561,Just Herbs Party Ready Nail Paint Kit(Packet)1...,1385,Marico Ltd.,5271.0,Marico Ltd,11,2026-09-30


In [647]:
chain_forecast_unpivoted['chain_name'] = 'Blinkit'

In [225]:
final_df = pd.read_excel('/data/aman_singh/acuuracy_check/Qcom_chain_depot_psku_forecast_as_on_5th_june_2026.xlsx', sheet_name = 'Base')
final_df

,Key,Chain,Depot,PSKU,PSKU Description,Brand,Index Rate,Portfolio,Run Month,Month Date,...,Primary_chain_value,Calculated Primary Val_from_chain_psku_offtakes,Calculated Primary Vol_from_chain_psku_offtakes,chain_accuracy_flag,Chain_diff_from_chain_psku_offtakes_model,Chain_diff_from_chain_depot_psku_offtakes_model,Brand class,NPD Tag,Planning Principle,Primary P3M 0?
0,blinkit_d112_718287,Blinkit,d112,718287,PCNO 200ml JAR,PCNO(R),349274.001420,CNO,2026-06-30,2026-07-31,...,0.000000,0.0,0.0,0,0.000000,0.000000,A,NON-NPD,Valid,True
1,blinkit_d112_718287,Blinkit,d112,718287,PCNO 200ml JAR,PCNO(R),349274.001420,CNO,2026-06-30,2026-08-31,...,0.000000,0.0,0.0,0,0.000000,0.000000,A,NON-NPD,Valid,True
2,blinkit_d112_718287,Blinkit,d112,718287,PCNO 200ml JAR,PCNO(R),349274.001420,CNO,2026-06-30,2026-09-30,...,0.000000,0.0,0.0,0,0.000000,0.000000,A,NON-NPD,Valid,True
3,blinkit_d112_718288,Blinkit,d112,718288,SAFF GOLD 5L JAR,SAFF GOLD,138865.260689,Saffola Oils,2026-06-30,2026-07-31,...,0.000583,0.0,0.0,1,0.000583,0.000583,A,NON-NPD,Valid,True
4,blinkit_d112_718288,Blinkit,d112,718288,SAFF GOLD 5L JAR,SAFF GOLD,138865.260689,Saffola Oils,2026-06-30,2026-08-31,...,0.000583,0.0,0.0,1,0.000583,0.000583,A,NON-NPD,Valid,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63388,zepto_d674_811269,Zepto,d674,811269,PA ENCALYPTUS ESS OIL 14ML,PA_ESS_HO,12860.631072,Hair Oils,2026-06-30,2026-08-31,...,0.000000,0.0,0.0,0,0.000000,0.000000,NPD,NPD,Valid,True
63389,zepto_d674_811269,Zepto,d674,811269,PA ENCALYPTUS ESS OIL 14ML,PA_ESS_HO,12860.631072,Hair Oils,2026-06-30,2026-09-30,...,0.000000,0.0,0.0,0,0.000000,0.000000,NPD,NPD,Valid,True
63390,zepto_d674_811279,Zepto,d674,811279,SAFFOLA COLDPRESS SSO GROUNDNUT 2L,SAF_CDPRS,260000.000000,Saffola Oils,2026-06-30,2026-07-31,...,0.000000,0.0,0.0,0,0.000000,0.000000,NPD,NON-NPD,Valid,True
63391,zepto_d674_811279,Zepto,d674,811279,SAFFOLA COLDPRESS SSO GROUNDNUT 2L,SAF_CDPRS,260000.000000,Saffola Oils,2026-06-30,2026-08-31,...,0.000000,0.0,0.0,0,0.000000,0.000000,NPD,NON-NPD,Valid,True


In [226]:
mmonth_ot_df

,month_date,key,chain,depot,parent_material_code,material_group_code,vol_in_roum,imputed,run_month
0,2026-03-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.0,1,2026-06-30
1,2026-04-30,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.0,1,2026-06-30
2,2026-05-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.0,1,2026-06-30
3,2026-03-31,blinkit_D112_718310,blinkit,D112,718310,PCNO(R),0.0,1,2026-06-30
4,2026-04-30,blinkit_D112_718310,blinkit,D112,718310,PCNO(R),0.0,1,2026-06-30
...,...,...,...,...,...,...,...,...,...
32404,2026-04-30,zepto_D677_810407,zepto,D677,810407,PURSNS_ML,0.0,1,2026-06-30
32405,2026-05-31,zepto_D677_810407,zepto,D677,810407,PURSNS_ML,0.0,1,2026-06-30
32406,2026-03-31,zepto_D677_810439,zepto,D677,810439,SAF-MUSLI,0.0,1,2026-06-30
32407,2026-04-30,zepto_D677_810439,zepto,D677,810439,SAF-MUSLI,0.0,1,2026-06-30


In [227]:
df1 = final_df.copy()
df2 = mmonth_ot_df.copy()

In [228]:
df1['key'] = df1['Key'].str.lower()
df2['key'] = df2['key'].str.lower()

df2['month_date'] = pd.to_datetime(df2['month_date'])

In [229]:
df2['month'] = df2['month_date'].dt.month
df2 = df2[df2['month'].isin([3, 4, 5])]
df2_pivot = df2.pivot_table(
    index='key',
    columns='month',
    values='vol_in_roum',
    aggfunc='sum'
).reset_index()

In [230]:
df2_pivot.rename(columns={
    3: 'offtakes_Mar_vol',
    4: 'offtakes_Apr_vol',
    5: 'offtakes_May_vol'
}, inplace=True)
df2_pivot

month,key,offtakes_Mar_vol,offtakes_Apr_vol,offtakes_May_vol
0,blinkit_d112_718288,0.000,0.00,0.00
1,blinkit_d112_718310,0.000,0.00,0.00
2,blinkit_d112_718312,0.817,0.82,1.34
3,blinkit_d112_718317,0.000,0.00,0.00
4,blinkit_d112_718319,0.000,0.00,0.00
...,...,...,...,...
10986,zepto_d677_810372,0.000,0.00,0.00
10987,zepto_d677_810405,0.000,0.00,0.00
10988,zepto_d677_810406,0.000,0.00,0.00
10989,zepto_d677_810407,0.000,0.00,0.00


In [231]:
df1.shape

(63393, 129)

In [232]:
df1 = df1.merge(df2_pivot, on='key', how='left')
df1

,Key,Chain,Depot,PSKU,PSKU Description,Brand,Index Rate,Portfolio,Run Month,Month Date,...,Chain_diff_from_chain_psku_offtakes_model,Chain_diff_from_chain_depot_psku_offtakes_model,Brand class,NPD Tag,Planning Principle,Primary P3M 0?,key,offtakes_Mar_vol,offtakes_Apr_vol,offtakes_May_vol
0,blinkit_d112_718287,Blinkit,d112,718287,PCNO 200ml JAR,PCNO(R),349274.001420,CNO,2026-06-30,2026-07-31,...,0.000000,0.000000,A,NON-NPD,Valid,True,blinkit_d112_718287,NaN,NaN,NaN
1,blinkit_d112_718287,Blinkit,d112,718287,PCNO 200ml JAR,PCNO(R),349274.001420,CNO,2026-06-30,2026-08-31,...,0.000000,0.000000,A,NON-NPD,Valid,True,blinkit_d112_718287,NaN,NaN,NaN
2,blinkit_d112_718287,Blinkit,d112,718287,PCNO 200ml JAR,PCNO(R),349274.001420,CNO,2026-06-30,2026-09-30,...,0.000000,0.000000,A,NON-NPD,Valid,True,blinkit_d112_718287,NaN,NaN,NaN
3,blinkit_d112_718288,Blinkit,d112,718288,SAFF GOLD 5L JAR,SAFF GOLD,138865.260689,Saffola Oils,2026-06-30,2026-07-31,...,0.000583,0.000583,A,NON-NPD,Valid,True,blinkit_d112_718288,0.0,0.0,0.0
4,blinkit_d112_718288,Blinkit,d112,718288,SAFF GOLD 5L JAR,SAFF GOLD,138865.260689,Saffola Oils,2026-06-30,2026-08-31,...,0.000583,0.000583,A,NON-NPD,Valid,True,blinkit_d112_718288,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63388,zepto_d674_811269,Zepto,d674,811269,PA ENCALYPTUS ESS OIL 14ML,PA_ESS_HO,12860.631072,Hair Oils,2026-06-30,2026-08-31,...,0.000000,0.000000,NPD,NPD,Valid,True,zepto_d674_811269,NaN,NaN,NaN
63389,zepto_d674_811269,Zepto,d674,811269,PA ENCALYPTUS ESS OIL 14ML,PA_ESS_HO,12860.631072,Hair Oils,2026-06-30,2026-09-30,...,0.000000,0.000000,NPD,NPD,Valid,True,zepto_d674_811269,NaN,NaN,NaN
63390,zepto_d674_811279,Zepto,d674,811279,SAFFOLA COLDPRESS SSO GROUNDNUT 2L,SAF_CDPRS,260000.000000,Saffola Oils,2026-06-30,2026-07-31,...,0.000000,0.000000,NPD,NON-NPD,Valid,True,zepto_d674_811279,NaN,NaN,NaN
63391,zepto_d674_811279,Zepto,d674,811279,SAFFOLA COLDPRESS SSO GROUNDNUT 2L,SAF_CDPRS,260000.000000,Saffola Oils,2026-06-30,2026-08-31,...,0.000000,0.000000,NPD,NON-NPD,Valid,True,zepto_d674_811279,NaN,NaN,NaN


In [234]:
df2_pivot.columns

Index(['key', 'offtakes_Mar_vol', 'offtakes_Apr_vol', 'offtakes_May_vol'], dtype='object', name='month')

In [235]:
vol_to_val_cols = ['offtakes_Mar_vol', 'offtakes_Apr_vol', 'offtakes_May_vol']
for col in vol_to_val_cols:
    df1[col[:-3] + 'Val'] = df1[col] * df1['Index Rate'] / (10 ** 7)

In [239]:
df1

,Key,Chain,Depot,PSKU,PSKU Description,Brand,Index Rate,Portfolio,Run Month,Month Date,...,NPD Tag,Planning Principle,Primary P3M 0?,key,offtakes_Mar_vol,offtakes_Apr_vol,offtakes_May_vol,offtakes_Mar_Val,offtakes_Apr_Val,offtakes_May_Val
0,blinkit_d112_718287,Blinkit,d112,718287,PCNO 200ml JAR,PCNO(R),349274.001420,CNO,2026-06-30,2026-07-31,...,NON-NPD,Valid,True,blinkit_d112_718287,NaN,NaN,NaN,NaN,NaN,NaN
1,blinkit_d112_718287,Blinkit,d112,718287,PCNO 200ml JAR,PCNO(R),349274.001420,CNO,2026-06-30,2026-08-31,...,NON-NPD,Valid,True,blinkit_d112_718287,NaN,NaN,NaN,NaN,NaN,NaN
2,blinkit_d112_718287,Blinkit,d112,718287,PCNO 200ml JAR,PCNO(R),349274.001420,CNO,2026-06-30,2026-09-30,...,NON-NPD,Valid,True,blinkit_d112_718287,NaN,NaN,NaN,NaN,NaN,NaN
3,blinkit_d112_718288,Blinkit,d112,718288,SAFF GOLD 5L JAR,SAFF GOLD,138865.260689,Saffola Oils,2026-06-30,2026-07-31,...,NON-NPD,Valid,True,blinkit_d112_718288,0.0,0.0,0.0,0.0,0.0,0.0
4,blinkit_d112_718288,Blinkit,d112,718288,SAFF GOLD 5L JAR,SAFF GOLD,138865.260689,Saffola Oils,2026-06-30,2026-08-31,...,NON-NPD,Valid,True,blinkit_d112_718288,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63388,zepto_d674_811269,Zepto,d674,811269,PA ENCALYPTUS ESS OIL 14ML,PA_ESS_HO,12860.631072,Hair Oils,2026-06-30,2026-08-31,...,NPD,Valid,True,zepto_d674_811269,NaN,NaN,NaN,NaN,NaN,NaN
63389,zepto_d674_811269,Zepto,d674,811269,PA ENCALYPTUS ESS OIL 14ML,PA_ESS_HO,12860.631072,Hair Oils,2026-06-30,2026-09-30,...,NPD,Valid,True,zepto_d674_811269,NaN,NaN,NaN,NaN,NaN,NaN
63390,zepto_d674_811279,Zepto,d674,811279,SAFFOLA COLDPRESS SSO GROUNDNUT 2L,SAF_CDPRS,260000.000000,Saffola Oils,2026-06-30,2026-07-31,...,NON-NPD,Valid,True,zepto_d674_811279,NaN,NaN,NaN,NaN,NaN,NaN
63391,zepto_d674_811279,Zepto,d674,811279,SAFFOLA COLDPRESS SSO GROUNDNUT 2L,SAF_CDPRS,260000.000000,Saffola Oils,2026-06-30,2026-08-31,...,NON-NPD,Valid,True,zepto_d674_811279,NaN,NaN,NaN,NaN,NaN,NaN


In [242]:
df1.groupby(['Month Date'])['offtakes_Mar_Val'].sum()

Month Date
2026-07-31    34.849015
2026-08-31    34.849015
2026-09-30    34.849015
Name: offtakes_Mar_Val, dtype: float64

In [243]:
df1.to_csv('Qcom_chain_depot_psku_2.csv')

In [1766]:
last_date_soh.groupby(['as_on_date'])['current_soh'].sum().reset_index()

,as_on_date,current_soh
0,2025-07-31,66300.094163
1,2025-08-31,58863.842909
2,2025-09-30,49363.157767
3,2025-10-31,72771.708935
4,2025-11-30,70224.268203
5,2025-12-31,96184.463361
6,2026-01-31,73347.849002
7,2026-02-28,77796.414711
8,2026-03-31,71152.461181


In [ ]:
query = """SELECT 
        * FROM
        dev_db.data_science.trn_soh_dc_master"""
dcm_table = pd.read_sql(
    query,
    dev_conn
)
dcm_table

import pandas as pd

# Build the DataFrame with Marico Depot split into code & city
rows = [
    {"CHAIN": "Grofers", "distributor_code": 16648, "sec_ind_bpm_mth (mesr-annual_ind_rate)": "85,75,980.43", "FC": "Kolkata K6 - Feeder Warehouse", "marico_depot_code": "D231", "marico_depot_city": "Kolkata 1"},
    {"CHAIN": "Grofers", "distributor_code": 18377, "sec_ind_bpm_mth (mesr-annual_ind_rate)": "9,99,319.43",  "FC": "Bengaluru B5 - Feeder Warehouse", "marico_depot_code": "D673", "marico_depot_city": "Bangalore"},
    {"CHAIN": "Grofers", "distributor_code": 16950, "sec_ind_bpm_mth (mesr-annual_ind_rate)": "3,17,265.52",  "FC": "Faridabad - Feeder Warehouse", "marico_depot_code": "D115", "marico_depot_city": "Sonipat"},
    {"CHAIN": "Swiggy",  "distributor_code": 16914, "sec_ind_bpm_mth (mesr-annual_ind_rate)": "32,60,801.69", "FC": "AMD IM3", "marico_depot_code": "D356", "marico_depot_city": "Bhiwandi II"},
    {"CHAIN": "Swiggy",  "distributor_code": 18466, "sec_ind_bpm_mth (mesr-annual_ind_rate)": "21,01,460.70", "FC": "HYD IM4", "marico_depot_code": "D530", "marico_depot_city": "HYDERABAD DEPOT"},
    {"CHAIN": "Swiggy",  "distributor_code": 16994, "sec_ind_bpm_mth (mesr-annual_ind_rate)": "15,84,475.14", "FC": "NAG IM1", "marico_depot_code": "D356", "marico_depot_city": "Bhiwandi II"},
    {"CHAIN": "Zepto",   "distributor_code": 16808, "sec_ind_bpm_mth (mesr-annual_ind_rate)": "1,70,56,520.12", "FC": "BLR-DRY-MH-Sumadhura2", "marico_depot_code": "D673", "marico_depot_city": "Bangalore"},
    # Last row had FC, then depot code (D113) on next line, then city (Lucknow) on the following line
    {"CHAIN": "Zepto",   "distributor_code": 16804, "sec_ind_bpm_mth (mesr-annual_ind_rate)": "1,35,66,121.70", "FC": "LKO-DRY-MH-SOHRAMAU", "marico_depot_code": "D113", "marico_depot_city": "Lucknow"},
]

df = pd.DataFrame(rows)

# Clean Indian-formatted numbers -> float
num_col = "sec_ind_bpm_mth (mesr-annual_ind_rate)"
df[num_col] = df[num_col].str.replace(",", "", regex=False).astype(float)

print(df)
print("\nData types:\n", df.dtypes)
dcm_table.columns
#df.drop(columns=['sec_ind_bpm_mth (mesr-annual_ind_rate)'], inplace=True)
df.rename(columns={'distributor_code': 'customer','FC':'facility_name', 'marico_depot_city':'city',
                   'marico_depot_code':'marico_depot','CHAIN':'channel'}, inplace=True)
df

df['status'] = 'None'
df['depot_name'] = 'None'
df
df.columns = df.columns.str.upper()
df
dcm_table['channel'].unique()
dcm_table.columns = dcm_table.columns.str.lower()
dcm_table[dcm_table['customer'].isin(df['distributor_code'].unique())]
df['CHANNEL'].unique()
df["CHANNEL"] = df["CHANNEL"].replace("Grofers", "Blinkit")
df
dcm_table.columns = dcm_table.columns.str.upper()
df['STATUS'] = 'Active'
df2 = pd.concat([dcm_table, df], ignore_index=True)
df2

df2.to_csv('trn_soh_dc_master.csv', index=False)
from snowflake.connector.pandas_tools import write_pandas

write_pandas(dev_conn, df2, 
            table_name = "TRN_SOH_DC_MASTER",
            auto_create_table=True,
            overwrite = False,
)

In [1443]:
df.to_csv('city____.csv')